# CNN KuLSIF density-ratio estimation

This notebook preserves the original patch-size and convolution experiment
variants. Each large experiment cell is self-contained; run only the section
you intend to reproduce. Generated outputs have been removed for a clean Git
history.

Run the configuration cell first. Override its paths with the
`DRE_PROJECT_ROOT`, `DRE_DATA_ROOT`, `DRE_OUTPUT_ROOT`, `DRE_RASTER_DIR`, or
`DRE_LANDMASK_PATH` environment variables when needed.


In [ ]:
from pathlib import Path
import os

PROJECT_ROOT = Path(os.environ.get("DRE_PROJECT_ROOT", Path.cwd())).expanduser().resolve()
DATA_ROOT = Path(os.environ.get("DRE_DATA_ROOT", PROJECT_ROOT / "data" / "processed")).expanduser().resolve()
OUTPUT_ROOT = Path(os.environ.get("DRE_OUTPUT_ROOT", PROJECT_ROOT / "outputs" / "dre_approaches")).expanduser().resolve()
RASTER_DIR = Path(os.environ.get("DRE_RASTER_DIR", DATA_ROOT / "covariate_rasters")).expanduser().resolve()
LANDMASK_PATH = Path(os.environ.get("DRE_LANDMASK_PATH", DATA_ROOT / "landmask.tif")).expanduser().resolve()

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
print("Project root:", PROJECT_ROOT)
print("Data root:", DATA_ROOT)
print("Output root:", OUTPUT_ROOT)


# Patch size 3


In [ ]:
import os
from typing import List, Dict, Tuple

import numpy as np

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW

from sklearn.metrics import roc_auc_score

# =========================================================
# CONFIG
# =========================================================

CV_DIR = str(DATA_ROOT / "cv_patches_3")
TEST_DIR = str(DATA_ROOT / "test_patches_3")

CV_X_PATH = os.path.join(CV_DIR, "X.npy")
CV_M_PATH = os.path.join(CV_DIR, "M.npy")
CV_Y_PATH = os.path.join(CV_DIR, "y.npy")
CV_FOLD_PATH = os.path.join(CV_DIR, "fold.npy")

TEST_X_PATH = os.path.join(TEST_DIR, "X.npy")
TEST_M_PATH = os.path.join(TEST_DIR, "M.npy")
TEST_Y_PATH = os.path.join(TEST_DIR, "y.npy")

MODEL_DIR = str(OUTPUT_ROOT / "kulsif" / "cnn_kulsif_patch_models_3")

BATCH_SIZE = 256
NUM_WORKERS = 0

MAX_EPOCHS = 100
PATIENCE = 10

LR = 1e-3
WEIGHT_DECAY = 1e-3

SEED = 42

EMB_DIM = 32
HIDDEN_DIMS = [32]
DROPOUT = 0.35

PATCH_SIZE = 3  # 13x13 patches

# Data augmentation
USE_FLIPS = True
NOISE_STD = 0.08

# Model selection gate (optional): computed on bounded monotone score01 = w/(1+w)
USE_AUC_FLOOR = False
AUC_FLOOR = 0.80

# Numeric stability
W_EPS = 1e-8
MAX_W = 1e6

# KuLSIF penalty strength (optional): encourages E_q[w] = 1
KULSIF_NORM_LAM = 10.0

# If your labels are reversed, flip these:
#   y==1 -> p (target)
#   y==0 -> q (reference)
P_LABEL = 1
Q_LABEL = 0


# =========================================================
# Device
# =========================================================
if torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


# =========================================================
# Metrics: Boyce + AUC (expects any real-valued score)
# =========================================================
def _average_ranks(x: np.ndarray) -> np.ndarray:
    x = np.asarray(x, dtype=float)
    order = np.argsort(x, kind="mergesort")
    ranks = np.empty_like(order, dtype=float)
    ranks[order] = np.arange(1, len(x) + 1, dtype=float)
    sx = x[order]
    i = 0
    while i < len(x):
        j = i + 1
        while j < len(x) and sx[j] == sx[i]:
            j += 1
        if j - i > 1:
            avg = (i + 1 + j) / 2.0
            ranks[order[i:j]] = avg
        i = j
    return ranks


def _spearman_tieaware(x, y) -> float:
    x = np.asarray(x, float)
    y = np.asarray(y, float)
    if x.size < 3 or y.size < 3:
        return np.nan
    rx, ry = _average_ranks(x), _average_ranks(y)
    if np.all(rx == rx[0]) or np.all(ry == ry[0]):
        return np.nan
    return float(np.corrcoef(rx, ry)[0, 1])


def continuous_boyce(y_true, scores, nbins_max=20, min_per_group=10):
    y = np.asarray(y_true).astype(int)
    s = np.asarray(scores, dtype=float)

    s_bg = s[y == 0]
    s_pr = s[y == 1]
    if (s_bg.size < min_per_group) or (s_pr.size < min_per_group):
        return np.nan

    uq = np.unique(s_bg[np.isfinite(s_bg)])
    if uq.size < 3:
        return np.nan
    nb = min(nbins_max, max(3, uq.size - 1))

    qs = np.quantile(s_bg, np.linspace(0.0, 1.0, nb + 1))
    qs[0] -= 1e-12
    qs[-1] += 1e-12

    pratio, centers = [], []
    Lb, Lp = float(len(s_bg)), float(len(s_pr))
    for a, b in zip(qs[:-1], qs[1:]):
        in_bg = (s_bg >= a) & (s_bg < b)
        nbk = int(in_bg.sum())
        if nbk == 0:
            continue
        in_pr = (s_pr >= a) & (s_pr < b)
        npk = int(in_pr.sum())
        pratio.append((npk / Lp) / (nbk / Lb))
        centers.append(0.5 * (a + b))

    if len(pratio) < 3:
        return np.nan
    return _spearman_tieaware(np.asarray(centers), np.asarray(pratio))


def compute_auc(y_true, scores):
    y = np.asarray(y_true, int)
    s = np.asarray(scores, float)
    m = np.isfinite(s)
    y, s = y[m], s[m]
    if y.size < 2 or np.unique(y).size < 2:
        return np.nan
    return float(roc_auc_score(y, s))


def compute_boyce_and_auc(y_true, scores, nbins_boyce=20):
    y = np.asarray(y_true, int)
    s = np.asarray(scores, float)
    m = np.isfinite(s)
    y, s = y[m], s[m]
    if y.size == 0:
        return dict(Boyce=np.nan, ROC_AUC=np.nan)
    return dict(
        Boyce=continuous_boyce(y, s, nbins_max=nbins_boyce),
        ROC_AUC=compute_auc(y, s),
    )


# =========================================================
# Helpers
# =========================================================
def w_to_score01_np(w: np.ndarray) -> np.ndarray:
    w = np.asarray(w, dtype=np.float64)
    w = np.clip(w, 0.0, MAX_W)
    return w / (1.0 + w)


def set_seed(seed=42):
    import random
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def make_cv_indices(fold_cv: np.ndarray) -> List[int]:
    vals = np.unique(fold_cv)
    return sorted(int(v) for v in vals)


# =========================================================
# Dataset + Models
# =========================================================
def augment_patch(values: torch.Tensor, mask: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
    if USE_FLIPS:
        if torch.rand(1).item() < 0.5:
            values = torch.flip(values, dims=[1])
            mask = torch.flip(mask, dims=[1])
        if torch.rand(1).item() < 0.5:
            values = torch.flip(values, dims=[2])
            mask = torch.flip(mask, dims=[2])

    if NOISE_STD > 0.0:
        noise = torch.randn_like(values) * NOISE_STD
        values = values + noise

    return values, mask


class PatchDREDataset(Dataset):
    """Returns xv, xm, y. Useful for validation metrics only."""
    def __init__(self, X_values, X_masks, y, indices, train: bool):
        super().__init__()
        self.Xv = X_values.astype(np.float32)
        self.Xm = X_masks.astype(np.float32)
        self.y = y.astype(np.float32)
        self.indices = np.asarray(indices, dtype=np.int64)
        self.train = train

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i: int):
        idx = self.indices[i]
        xv = torch.from_numpy(self.Xv[idx])
        xm = torch.from_numpy(self.Xm[idx])
        y = torch.tensor(self.y[idx], dtype=torch.float32)
        if self.train:
            xv, xm = augment_patch(xv, xm)
        return xv, xm, y


class PatchXDataset(Dataset):
    """Returns xv, xm only. Used for p/q loaders."""
    def __init__(self, X_values, X_masks, indices, train: bool):
        super().__init__()
        self.Xv = X_values.astype(np.float32)
        self.Xm = X_masks.astype(np.float32)
        self.indices = np.asarray(indices, dtype=np.int64)
        self.train = train

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i: int):
        idx = self.indices[i]
        xv = torch.from_numpy(self.Xv[idx])
        xm = torch.from_numpy(self.Xm[idx])
        if self.train:
            xv, xm = augment_patch(xv, xm)
        return xv, xm


class MLP(nn.Module):
    def __init__(self, in_dim: int, hidden_dims=(256, 128, 64), dropout=0.2):
        super().__init__()
        layers = []
        prev = in_dim
        for h in hidden_dims:
            layers.append(nn.Linear(prev, h))
            layers.append(nn.ReLU())
            if dropout > 0:
                layers.append(nn.Dropout(dropout))
            prev = h
        layers.append(nn.Linear(prev, 1))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x).squeeze(-1)


class PatchEncoder13(nn.Module):
    def __init__(self, in_value_channels: int, emb_dim: int = 32):
        super().__init__()
        in_channels = 2 * in_value_channels
        self.features = nn.Sequential(
            nn.Conv2d(in_channels, 16, 3, padding=1, bias=False),
            nn.BatchNorm2d(16),
            nn.ReLU(inplace=True),

            nn.Conv2d(16, 16, 3, padding=1, bias=False),
            nn.BatchNorm2d(16),
            nn.ReLU(inplace=True),

            nn.MaxPool2d(2, 2),  # 13→6

            nn.Conv2d(16, 32, 3, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),

            nn.AdaptiveAvgPool2d(1),
        )
        self.proj = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32, emb_dim),
            nn.ReLU(inplace=True),
        )

    def forward(self, x_val, x_mask):
        x_val = x_val * x_mask
        x_in = torch.cat([x_val, x_mask], dim=1)
        h = self.features(x_in)
        z = self.proj(h)
        return z


class CNNKuLSIF(nn.Module):
    """
    Neural KuLSIF/uLSIF: network outputs w(x) >= 0 (density ratio).
    Parameterization: w = softplus(raw) + eps.
    """
    def __init__(self, in_value_channels, emb_dim=64, hidden_dims=(128, 64), dropout=0.2,
                 max_w=1e6, eps=1e-8):
        super().__init__()
        self.encoder = PatchEncoder13(in_value_channels, emb_dim)
        self.head = MLP(in_dim=emb_dim, hidden_dims=hidden_dims, dropout=dropout)
        self.max_w = float(max_w) if max_w is not None else None
        self.eps = float(eps)

    def forward(self, x_val, x_mask):
        z = self.encoder(x_val, x_mask)
        raw = self.head(z)
        w = torch.nn.functional.softplus(raw) + self.eps
        if self.max_w is not None:
            w = torch.clamp(w, 0.0, self.max_w)
        return w


# =========================================================
# KuLSIF loss:
# minimize 0.5 E_q[w^2] - E_p[w]  (+ optional penalty for E_q[w]=1)
# =========================================================
def kulsif_loss(w_p: torch.Tensor, w_q: torch.Tensor, norm_lam: float) -> Tuple[torch.Tensor, float]:
    loss = 0.5 * (w_q ** 2).mean() - w_p.mean()
    norm_err = (w_q.mean() - 1.0)
    if norm_lam is not None and norm_lam > 0:
        loss = loss + norm_lam * (norm_err ** 2)
    return loss, float(norm_err.detach().cpu())


# =========================================================
# Training per fold (KuLSIF)
# - train on p/q batches
# - validation metrics computed on bounded score01 = w/(1+w)
# - returns OOF bounded score01 for that fold val split
# =========================================================
def train_and_save_fold_model_kulsif_cnn(
    fold_id: int,
    X_values: np.ndarray,
    X_masks: np.ndarray,
    y_cv: np.ndarray,
    fold_cv: np.ndarray,
    model_dir: str,
) -> Tuple[Dict[str, float], np.ndarray, np.ndarray]:
    os.makedirs(model_dir, exist_ok=True)

    f = fold_cv.astype(int)
    train_idx = np.where(f != fold_id)[0]
    val_idx = np.where(f == fold_id)[0]

    # p/q split inside train
    p_tr = train_idx[y_cv[train_idx] == P_LABEL]
    q_tr = train_idx[y_cv[train_idx] == Q_LABEL]
    p_va = val_idx[y_cv[val_idx] == P_LABEL]
    q_va = val_idx[y_cv[val_idx] == Q_LABEL]

    if len(p_tr) < 10 or len(q_tr) < 10:
        raise RuntimeError(f"[fold {fold_id}] too few samples: p_tr={len(p_tr)} q_tr={len(q_tr)}")

    print(f"\n[fold {fold_id}] train: p={len(p_tr)} q={len(q_tr)} | val: p={len(p_va)} q={len(q_va)}")

    ds_p_tr = PatchXDataset(X_values, X_masks, p_tr, train=True)
    ds_q_tr = PatchXDataset(X_values, X_masks, q_tr, train=True)
    ds_val = PatchDREDataset(X_values, X_masks, y_cv, val_idx, train=False)

    dl_p_tr = DataLoader(ds_p_tr, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, drop_last=True)
    dl_q_tr = DataLoader(ds_q_tr, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, drop_last=True)
    dl_val = DataLoader(ds_val, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, drop_last=False)

    in_value_channels = X_values.shape[1]
    model = CNNKuLSIF(
        in_value_channels=in_value_channels,
        emb_dim=EMB_DIM,
        hidden_dims=HIDDEN_DIMS,
        dropout=DROPOUT,
        max_w=MAX_W,
        eps=W_EPS,
    ).to(device)

    opt = AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

    def run_train_epoch():
        model.train()
        total_loss = 0.0
        total_steps = 0
        norm_err_avg = 0.0

        for (xvp, xmp), (xvq, xmq) in zip(dl_p_tr, dl_q_tr):
            xvp, xmp = xvp.to(device), xmp.to(device)
            xvq, xmq = xvq.to(device), xmq.to(device)

            opt.zero_grad(set_to_none=True)

            w_p = model(xvp, xmp)
            w_q = model(xvq, xmq)

            loss, ne = kulsif_loss(w_p, w_q, norm_lam=KULSIF_NORM_LAM)

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()

            total_loss += float(loss.detach().cpu())
            norm_err_avg += ne
            total_steps += 1

        return {
            "kulsif_loss": total_loss / max(1, total_steps),
            "norm_err": norm_err_avg / max(1, total_steps),
        }

    def run_val_metrics():
        model.eval()
        all_w, all_y = [], []
        with torch.no_grad():
            for xv, xm, yb in dl_val:
                xv, xm = xv.to(device), xm.to(device)
                w = model(xv, xm)
                all_w.append(w.detach().cpu().numpy())
                all_y.append(yb.numpy())

        w_np = np.concatenate(all_w)
        y_np = np.concatenate(all_y).astype(int)

        score01 = w_to_score01_np(w_np)
        mets = compute_boyce_and_auc(y_np, score01)
        return mets, score01, w_np, y_np

    candidates = []
    best_boyce_key = -np.inf
    no_improve = 0

    for ep in range(1, MAX_EPOCHS + 1):
        tr = run_train_epoch()
        va, val_score01, val_w, val_y = run_val_metrics()

        print(
            f"[fold {fold_id}] epoch {ep:03d} | "
            f"train loss {tr['kulsif_loss']:.4f}, norm_err {tr['norm_err']:.4f} | "
            f"val AUC {va['ROC_AUC']:.4f}, Boyce {va['Boyce']:.4f}"
        )

        auc_ok = True
        if USE_AUC_FLOOR:
            auc_ok = np.isfinite(va["ROC_AUC"]) and (va["ROC_AUC"] >= AUC_FLOOR)

        if auc_ok:
            state = {
                "model_state": {k: v.detach().cpu().clone() for k, v in model.state_dict().items()},
                "in_value_channels": in_value_channels,
                "patch_size": PATCH_SIZE,
                "emb_dim": EMB_DIM,
                "hidden_dims": HIDDEN_DIMS,
                "kulsif_norm_lam": float(KULSIF_NORM_LAM),
                "max_w": float(MAX_W),
                "w_eps": float(W_EPS),
                "p_label": int(P_LABEL),
                "q_label": int(Q_LABEL),
            }
            candidates.append(
                {
                    "epoch": ep,
                    "boyce": float(va["Boyce"]) if np.isfinite(va["Boyce"]) else np.nan,
                    "auc": float(va["ROC_AUC"]) if np.isfinite(va["ROC_AUC"]) else np.nan,
                    "state": state,
                    "val_score01": val_score01,
                    "val_w": val_w,
                    "val_y": val_y,
                }
            )

            b = candidates[-1]["boyce"]
            b_key = -np.inf if not np.isfinite(b) else float(b)
            if b_key > best_boyce_key + 1e-9:
                best_boyce_key = b_key
                no_improve = 0
            else:
                no_improve += 1
        else:
            no_improve += 1

        if no_improve >= PATIENCE:
            print(f"[fold {fold_id}] early stopping at epoch {ep}")
            break

    if len(candidates) == 0:
        raise RuntimeError(
            f"[fold {fold_id}] No epoch met AUC floor (AUC_FLOOR={AUC_FLOOR}). "
            f"Lower AUC_FLOOR or set USE_AUC_FLOOR=False."
        )

    def key(c):
        b = c["boyce"]
        a = c["auc"]
        b_val = -np.inf if not np.isfinite(b) else float(b)
        a_val = -np.inf if not np.isfinite(a) else float(a)
        return (b_val, a_val)

    best = max(candidates, key=key)

    ckpt_path = os.path.join(model_dir, f"cnn_kulsif_fold_{fold_id}.pt")
    torch.save(best["state"], ckpt_path)
    print(
        f"[fold {fold_id}] selected epoch={best['epoch']} | "
        f"Boyce={best['boyce']:.4f}, AUC={best['auc']:.4f} | saved -> {ckpt_path}"
    )

    best_val = {"Boyce": best["boyce"], "ROC_AUC": best["auc"], "loss": np.nan}
    return best_val, best["val_score01"], val_idx


# =========================================================
# Ensemble prediction (KuLSIF)
# - each model outputs w
# - ensemble = weighted mean of w
# - bounded score01 = w/(1+w)
# =========================================================
def ensemble_predict_on_indices_kulsif_cnn(
    X_values: np.ndarray,
    X_masks: np.ndarray,
    y: np.ndarray,
    indices: np.ndarray,
    model_dir: str,
    fold_ids: List[int],
    weights: np.ndarray,
):
    in_value_channels = X_values.shape[1]

    models = []
    for fid in fold_ids:
        ckpt_path = os.path.join(model_dir, f"cnn_kulsif_fold_{fid}.pt")
        state = torch.load(ckpt_path, map_location="cpu")

        if state["in_value_channels"] != in_value_channels:
            raise RuntimeError(f"Value channel mismatch for fold {fid}")
        if state.get("patch_size", PATCH_SIZE) != PATCH_SIZE:
            raise RuntimeError(f"Patch size mismatch for fold {fid}")

        model = CNNKuLSIF(
            in_value_channels=in_value_channels,
            emb_dim=state.get("emb_dim", EMB_DIM),
            hidden_dims=tuple(state.get("hidden_dims", HIDDEN_DIMS)),
            dropout=DROPOUT,
            max_w=state.get("max_w", MAX_W),
            eps=state.get("w_eps", W_EPS),
        ).to(device)
        model.load_state_dict(state["model_state"])
        model.eval()
        models.append(model)

    w = np.asarray(weights, dtype=np.float64)
    if (not np.isfinite(w).all()) or w.sum() <= 0:
        w = np.ones(len(fold_ids), dtype=np.float64) / len(fold_ids)
    else:
        w = w / w.sum()

    weights_t = torch.tensor(w, dtype=torch.float32, device=device)

    idx = np.asarray(indices, dtype=np.int64)
    all_score01, all_w, all_y = [], [], []

    with torch.no_grad():
        for start in range(0, len(idx), BATCH_SIZE):
            end = min(start + BATCH_SIZE, len(idx))
            idx_batch = idx[start:end]

            xv = torch.from_numpy(X_values[idx_batch].astype(np.float32)).to(device)
            xm = torch.from_numpy(X_masks[idx_batch].astype(np.float32)).to(device)
            yb = torch.from_numpy(y[idx_batch].astype(np.float32)).to(device)

            w_ens = torch.zeros(xv.size(0), device=device)
            for k, model in enumerate(models):
                w_k = model(xv, xm)
                w_ens += weights_t[k] * w_k

            w_ens = torch.clamp(w_ens, 0.0, MAX_W)
            score01 = w_ens / (1.0 + w_ens)

            all_score01.append(score01.cpu().numpy())
            all_w.append(w_ens.cpu().numpy())
            all_y.append(yb.cpu().numpy())

    return np.concatenate(all_score01), np.concatenate(all_w), np.concatenate(all_y)


# =========================================================
# MAIN
# =========================================================
if __name__ == "__main__":
    set_seed(SEED)
    os.makedirs(MODEL_DIR, exist_ok=True)

    # ----------- load CV data -----------
    X_cv = np.load(CV_X_PATH)
    M_cv = np.load(CV_M_PATH)
    y_cv = np.load(CV_Y_PATH).astype(np.float32)
    fold_cv_raw = np.load(CV_FOLD_PATH)

    print("[CV] X shape:", X_cv.shape)
    print("[CV] M shape:", M_cv.shape)
    print("[CV] y shape:", y_cv.shape)
    print("[CV] fold shape:", fold_cv_raw.shape)

    if X_cv.shape[0] != y_cv.shape[0] or M_cv.shape[0] != y_cv.shape[0]:
        raise ValueError("Mismatch between CV X/M and y lengths")

    if not np.isfinite(fold_cv_raw).all():
        raise ValueError("fold.npy contains NaN/inf; CV split must be defined for all rows")
    fold_cv = fold_cv_raw.astype(int)

    bad = ~np.isfinite(X_cv)
    if bad.any():
        print(f"[CV] Warning: {int(bad.sum())} non-finite X values set to 0.")
        X_cv[bad] = 0.0
    bad_m = ~np.isfinite(M_cv)
    if bad_m.any():
        print(f"[CV] Warning: {int(bad_m.sum())} non-finite M values set to 0.")
        M_cv[bad_m] = 0.0

    # ----------- load TEST data -----------
    X_test = np.load(TEST_X_PATH)
    M_test = np.load(TEST_M_PATH)
    y_test = np.load(TEST_Y_PATH).astype(np.float32)

    print("[TEST] X shape:", X_test.shape)
    print("[TEST] M shape:", M_test.shape)
    print("[TEST] y shape:", y_test.shape)

    if X_test.shape[0] != y_test.shape[0] or M_test.shape[0] != y_test.shape[0]:
        raise ValueError("Mismatch between TEST X/M and y lengths")

    bad = ~np.isfinite(X_test)
    if bad.any():
        print(f"[TEST] Warning: {int(bad.sum())} non-finite X values set to 0.")
        X_test[bad] = 0.0
    bad_m = ~np.isfinite(M_test)
    if bad_m.any():
        print(f"[TEST] Warning: {int(bad_m.sum())} non-finite M values set to 0.")
        M_test[bad_m] = 0.0

    # ----------- CV training over folds -----------
    fold_ids = make_cv_indices(fold_cv)
    print("Folds:", fold_ids)

    # OOF bounded scores (0..1) for metrics comparability
    oof_score01 = np.full_like(y_cv, np.nan, dtype=float)

    fold_aucs = []
    fold_boyces = []

    for fid in fold_ids:
        best_val, val_score01, val_idx = train_and_save_fold_model_kulsif_cnn(
            fid, X_cv, M_cv, y_cv, fold_cv, MODEL_DIR
        )
        fold_aucs.append(best_val["ROC_AUC"])
        fold_boyces.append(best_val["Boyce"])
        oof_score01[val_idx] = val_score01

    # ----------- CV metrics (OOF bounded score) -----------
    cv_metrics = compute_boyce_and_auc(y_cv, oof_score01)
    print("\n[CV] OOF metrics (bounded score = w/(1+w)):", cv_metrics)
    print("[CV] per-fold best AUCs:", fold_aucs)
    print("[CV] per-fold best Boyce:", fold_boyces)

    # ----------- Ensemble weights -----------
    # KuLSIF doesn't give a comparable "loss" like BCE, so default to equal weights.
    weights = np.ones(len(fold_ids), dtype=float) / len(fold_ids)
    print("\n[Ensemble] Using equal weights:", weights)

    np.save(os.path.join(MODEL_DIR, "fold_ids.npy"), np.array(fold_ids, dtype=int))
    np.save(os.path.join(MODEL_DIR, "fold_weights_equal.npy"), weights)
    np.save(os.path.join(MODEL_DIR, "oof_score01.npy"), oof_score01)
    print("[Saved] fold_ids.npy, fold_weights_equal.npy, oof_score01.npy")

    # ----------- External test ensemble (bounded score) -----------
    test_idx = np.arange(len(y_test))
    print("External test N =", len(test_idx))

    test_score01, test_w, test_y = ensemble_predict_on_indices_kulsif_cnn(
        X_values=X_test,
        X_masks=M_test,
        y=y_test,
        indices=test_idx,
        model_dir=MODEL_DIR,
        fold_ids=fold_ids,
        weights=weights,
    )

    test_metrics = compute_boyce_and_auc(test_y, test_score01)
    print("\n[Test] Ensemble metrics (bounded score = w/(1+w)):", test_metrics)

    np.save(os.path.join(MODEL_DIR, "test_score01.npy"), test_score01)
    np.save(os.path.join(MODEL_DIR, "test_w_ens.npy"), test_w)
    print("[Test] Saved test_score01.npy, test_w_ens.npy")


# Patch size: 5


In [ ]:
import os
from typing import List, Dict, Tuple

import numpy as np

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW

from sklearn.metrics import roc_auc_score

# =========================================================
# CONFIG
# =========================================================

CV_DIR = str(DATA_ROOT / "cv_patches_5")
TEST_DIR = str(DATA_ROOT / "test_patches_5")

CV_X_PATH = os.path.join(CV_DIR, "X.npy")
CV_M_PATH = os.path.join(CV_DIR, "M.npy")
CV_Y_PATH = os.path.join(CV_DIR, "y.npy")
CV_FOLD_PATH = os.path.join(CV_DIR, "fold.npy")

TEST_X_PATH = os.path.join(TEST_DIR, "X.npy")
TEST_M_PATH = os.path.join(TEST_DIR, "M.npy")
TEST_Y_PATH = os.path.join(TEST_DIR, "y.npy")

MODEL_DIR = str(OUTPUT_ROOT / "kulsif" / "cnn_kulsif_patch_models_5")

BATCH_SIZE = 256
NUM_WORKERS = 0

MAX_EPOCHS = 100
PATIENCE = 10

LR = 1e-3
WEIGHT_DECAY = 1e-3

SEED = 42

EMB_DIM = 32
HIDDEN_DIMS = [32]
DROPOUT = 0.35

PATCH_SIZE = 5  # 13x13 patches

# Data augmentation
USE_FLIPS = True
NOISE_STD = 0.08

# Model selection gate (optional): computed on bounded monotone score01 = w/(1+w)
USE_AUC_FLOOR = False
AUC_FLOOR = 0.80

# Numeric stability
W_EPS = 1e-8
MAX_W = 1e6

# KuLSIF penalty strength (optional): encourages E_q[w] = 1
KULSIF_NORM_LAM = 10.0

# If your labels are reversed, flip these:
#   y==1 -> p (target)
#   y==0 -> q (reference)
P_LABEL = 1
Q_LABEL = 0


# =========================================================
# Device
# =========================================================
if torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


# =========================================================
# Metrics: Boyce + AUC (expects any real-valued score)
# =========================================================
def _average_ranks(x: np.ndarray) -> np.ndarray:
    x = np.asarray(x, dtype=float)
    order = np.argsort(x, kind="mergesort")
    ranks = np.empty_like(order, dtype=float)
    ranks[order] = np.arange(1, len(x) + 1, dtype=float)
    sx = x[order]
    i = 0
    while i < len(x):
        j = i + 1
        while j < len(x) and sx[j] == sx[i]:
            j += 1
        if j - i > 1:
            avg = (i + 1 + j) / 2.0
            ranks[order[i:j]] = avg
        i = j
    return ranks


def _spearman_tieaware(x, y) -> float:
    x = np.asarray(x, float)
    y = np.asarray(y, float)
    if x.size < 3 or y.size < 3:
        return np.nan
    rx, ry = _average_ranks(x), _average_ranks(y)
    if np.all(rx == rx[0]) or np.all(ry == ry[0]):
        return np.nan
    return float(np.corrcoef(rx, ry)[0, 1])


def continuous_boyce(y_true, scores, nbins_max=20, min_per_group=10):
    y = np.asarray(y_true).astype(int)
    s = np.asarray(scores, dtype=float)

    s_bg = s[y == 0]
    s_pr = s[y == 1]
    if (s_bg.size < min_per_group) or (s_pr.size < min_per_group):
        return np.nan

    uq = np.unique(s_bg[np.isfinite(s_bg)])
    if uq.size < 3:
        return np.nan
    nb = min(nbins_max, max(3, uq.size - 1))

    qs = np.quantile(s_bg, np.linspace(0.0, 1.0, nb + 1))
    qs[0] -= 1e-12
    qs[-1] += 1e-12

    pratio, centers = [], []
    Lb, Lp = float(len(s_bg)), float(len(s_pr))
    for a, b in zip(qs[:-1], qs[1:]):
        in_bg = (s_bg >= a) & (s_bg < b)
        nbk = int(in_bg.sum())
        if nbk == 0:
            continue
        in_pr = (s_pr >= a) & (s_pr < b)
        npk = int(in_pr.sum())
        pratio.append((npk / Lp) / (nbk / Lb))
        centers.append(0.5 * (a + b))

    if len(pratio) < 3:
        return np.nan
    return _spearman_tieaware(np.asarray(centers), np.asarray(pratio))


def compute_auc(y_true, scores):
    y = np.asarray(y_true, int)
    s = np.asarray(scores, float)
    m = np.isfinite(s)
    y, s = y[m], s[m]
    if y.size < 2 or np.unique(y).size < 2:
        return np.nan
    return float(roc_auc_score(y, s))


def compute_boyce_and_auc(y_true, scores, nbins_boyce=20):
    y = np.asarray(y_true, int)
    s = np.asarray(scores, float)
    m = np.isfinite(s)
    y, s = y[m], s[m]
    if y.size == 0:
        return dict(Boyce=np.nan, ROC_AUC=np.nan)
    return dict(
        Boyce=continuous_boyce(y, s, nbins_max=nbins_boyce),
        ROC_AUC=compute_auc(y, s),
    )


# =========================================================
# Helpers
# =========================================================
def w_to_score01_np(w: np.ndarray) -> np.ndarray:
    w = np.asarray(w, dtype=np.float64)
    w = np.clip(w, 0.0, MAX_W)
    return w / (1.0 + w)


def set_seed(seed=42):
    import random
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def make_cv_indices(fold_cv: np.ndarray) -> List[int]:
    vals = np.unique(fold_cv)
    return sorted(int(v) for v in vals)


# =========================================================
# Dataset + Models
# =========================================================
def augment_patch(values: torch.Tensor, mask: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
    if USE_FLIPS:
        if torch.rand(1).item() < 0.5:
            values = torch.flip(values, dims=[1])
            mask = torch.flip(mask, dims=[1])
        if torch.rand(1).item() < 0.5:
            values = torch.flip(values, dims=[2])
            mask = torch.flip(mask, dims=[2])

    if NOISE_STD > 0.0:
        noise = torch.randn_like(values) * NOISE_STD
        values = values + noise

    return values, mask


class PatchDREDataset(Dataset):
    """Returns xv, xm, y. Useful for validation metrics only."""
    def __init__(self, X_values, X_masks, y, indices, train: bool):
        super().__init__()
        self.Xv = X_values.astype(np.float32)
        self.Xm = X_masks.astype(np.float32)
        self.y = y.astype(np.float32)
        self.indices = np.asarray(indices, dtype=np.int64)
        self.train = train

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i: int):
        idx = self.indices[i]
        xv = torch.from_numpy(self.Xv[idx])
        xm = torch.from_numpy(self.Xm[idx])
        y = torch.tensor(self.y[idx], dtype=torch.float32)
        if self.train:
            xv, xm = augment_patch(xv, xm)
        return xv, xm, y


class PatchXDataset(Dataset):
    """Returns xv, xm only. Used for p/q loaders."""
    def __init__(self, X_values, X_masks, indices, train: bool):
        super().__init__()
        self.Xv = X_values.astype(np.float32)
        self.Xm = X_masks.astype(np.float32)
        self.indices = np.asarray(indices, dtype=np.int64)
        self.train = train

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i: int):
        idx = self.indices[i]
        xv = torch.from_numpy(self.Xv[idx])
        xm = torch.from_numpy(self.Xm[idx])
        if self.train:
            xv, xm = augment_patch(xv, xm)
        return xv, xm


class MLP(nn.Module):
    def __init__(self, in_dim: int, hidden_dims=(256, 128, 64), dropout=0.2):
        super().__init__()
        layers = []
        prev = in_dim
        for h in hidden_dims:
            layers.append(nn.Linear(prev, h))
            layers.append(nn.ReLU())
            if dropout > 0:
                layers.append(nn.Dropout(dropout))
            prev = h
        layers.append(nn.Linear(prev, 1))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x).squeeze(-1)


class PatchEncoder13(nn.Module):
    def __init__(self, in_value_channels: int, emb_dim: int = 32):
        super().__init__()
        in_channels = 2 * in_value_channels
        self.features = nn.Sequential(
            nn.Conv2d(in_channels, 16, 3, padding=1, bias=False),
            nn.BatchNorm2d(16),
            nn.ReLU(inplace=True),

            nn.Conv2d(16, 16, 3, padding=1, bias=False),
            nn.BatchNorm2d(16),
            nn.ReLU(inplace=True),

            nn.MaxPool2d(2, 2),  # 13→6

            nn.Conv2d(16, 32, 3, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),

            nn.AdaptiveAvgPool2d(1),
        )
        self.proj = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32, emb_dim),
            nn.ReLU(inplace=True),
        )

    def forward(self, x_val, x_mask):
        x_val = x_val * x_mask
        x_in = torch.cat([x_val, x_mask], dim=1)
        h = self.features(x_in)
        z = self.proj(h)
        return z


class CNNKuLSIF(nn.Module):
    """
    Neural KuLSIF/uLSIF: network outputs w(x) >= 0 (density ratio).
    Parameterization: w = softplus(raw) + eps.
    """
    def __init__(self, in_value_channels, emb_dim=64, hidden_dims=(128, 64), dropout=0.2,
                 max_w=1e6, eps=1e-8):
        super().__init__()
        self.encoder = PatchEncoder13(in_value_channels, emb_dim)
        self.head = MLP(in_dim=emb_dim, hidden_dims=hidden_dims, dropout=dropout)
        self.max_w = float(max_w) if max_w is not None else None
        self.eps = float(eps)

    def forward(self, x_val, x_mask):
        z = self.encoder(x_val, x_mask)
        raw = self.head(z)
        w = torch.nn.functional.softplus(raw) + self.eps
        if self.max_w is not None:
            w = torch.clamp(w, 0.0, self.max_w)
        return w


# =========================================================
# KuLSIF loss:
# minimize 0.5 E_q[w^2] - E_p[w]  (+ optional penalty for E_q[w]=1)
# =========================================================
def kulsif_loss(w_p: torch.Tensor, w_q: torch.Tensor, norm_lam: float) -> Tuple[torch.Tensor, float]:
    loss = 0.5 * (w_q ** 2).mean() - w_p.mean()
    norm_err = (w_q.mean() - 1.0)
    if norm_lam is not None and norm_lam > 0:
        loss = loss + norm_lam * (norm_err ** 2)
    return loss, float(norm_err.detach().cpu())


# =========================================================
# Training per fold (KuLSIF)
# - train on p/q batches
# - validation metrics computed on bounded score01 = w/(1+w)
# - returns OOF bounded score01 for that fold val split
# =========================================================
def train_and_save_fold_model_kulsif_cnn(
    fold_id: int,
    X_values: np.ndarray,
    X_masks: np.ndarray,
    y_cv: np.ndarray,
    fold_cv: np.ndarray,
    model_dir: str,
) -> Tuple[Dict[str, float], np.ndarray, np.ndarray]:
    os.makedirs(model_dir, exist_ok=True)

    f = fold_cv.astype(int)
    train_idx = np.where(f != fold_id)[0]
    val_idx = np.where(f == fold_id)[0]

    # p/q split inside train
    p_tr = train_idx[y_cv[train_idx] == P_LABEL]
    q_tr = train_idx[y_cv[train_idx] == Q_LABEL]
    p_va = val_idx[y_cv[val_idx] == P_LABEL]
    q_va = val_idx[y_cv[val_idx] == Q_LABEL]

    if len(p_tr) < 10 or len(q_tr) < 10:
        raise RuntimeError(f"[fold {fold_id}] too few samples: p_tr={len(p_tr)} q_tr={len(q_tr)}")

    print(f"\n[fold {fold_id}] train: p={len(p_tr)} q={len(q_tr)} | val: p={len(p_va)} q={len(q_va)}")

    ds_p_tr = PatchXDataset(X_values, X_masks, p_tr, train=True)
    ds_q_tr = PatchXDataset(X_values, X_masks, q_tr, train=True)
    ds_val = PatchDREDataset(X_values, X_masks, y_cv, val_idx, train=False)

    dl_p_tr = DataLoader(ds_p_tr, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, drop_last=True)
    dl_q_tr = DataLoader(ds_q_tr, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, drop_last=True)
    dl_val = DataLoader(ds_val, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, drop_last=False)

    in_value_channels = X_values.shape[1]
    model = CNNKuLSIF(
        in_value_channels=in_value_channels,
        emb_dim=EMB_DIM,
        hidden_dims=HIDDEN_DIMS,
        dropout=DROPOUT,
        max_w=MAX_W,
        eps=W_EPS,
    ).to(device)

    opt = AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

    def run_train_epoch():
        model.train()
        total_loss = 0.0
        total_steps = 0
        norm_err_avg = 0.0

        for (xvp, xmp), (xvq, xmq) in zip(dl_p_tr, dl_q_tr):
            xvp, xmp = xvp.to(device), xmp.to(device)
            xvq, xmq = xvq.to(device), xmq.to(device)

            opt.zero_grad(set_to_none=True)

            w_p = model(xvp, xmp)
            w_q = model(xvq, xmq)

            loss, ne = kulsif_loss(w_p, w_q, norm_lam=KULSIF_NORM_LAM)

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()

            total_loss += float(loss.detach().cpu())
            norm_err_avg += ne
            total_steps += 1

        return {
            "kulsif_loss": total_loss / max(1, total_steps),
            "norm_err": norm_err_avg / max(1, total_steps),
        }

    def run_val_metrics():
        model.eval()
        all_w, all_y = [], []
        with torch.no_grad():
            for xv, xm, yb in dl_val:
                xv, xm = xv.to(device), xm.to(device)
                w = model(xv, xm)
                all_w.append(w.detach().cpu().numpy())
                all_y.append(yb.numpy())

        w_np = np.concatenate(all_w)
        y_np = np.concatenate(all_y).astype(int)

        score01 = w_to_score01_np(w_np)
        mets = compute_boyce_and_auc(y_np, score01)
        return mets, score01, w_np, y_np

    candidates = []
    best_boyce_key = -np.inf
    no_improve = 0

    for ep in range(1, MAX_EPOCHS + 1):
        tr = run_train_epoch()
        va, val_score01, val_w, val_y = run_val_metrics()

        print(
            f"[fold {fold_id}] epoch {ep:03d} | "
            f"train loss {tr['kulsif_loss']:.4f}, norm_err {tr['norm_err']:.4f} | "
            f"val AUC {va['ROC_AUC']:.4f}, Boyce {va['Boyce']:.4f}"
        )

        auc_ok = True
        if USE_AUC_FLOOR:
            auc_ok = np.isfinite(va["ROC_AUC"]) and (va["ROC_AUC"] >= AUC_FLOOR)

        if auc_ok:
            state = {
                "model_state": {k: v.detach().cpu().clone() for k, v in model.state_dict().items()},
                "in_value_channels": in_value_channels,
                "patch_size": PATCH_SIZE,
                "emb_dim": EMB_DIM,
                "hidden_dims": HIDDEN_DIMS,
                "kulsif_norm_lam": float(KULSIF_NORM_LAM),
                "max_w": float(MAX_W),
                "w_eps": float(W_EPS),
                "p_label": int(P_LABEL),
                "q_label": int(Q_LABEL),
            }
            candidates.append(
                {
                    "epoch": ep,
                    "boyce": float(va["Boyce"]) if np.isfinite(va["Boyce"]) else np.nan,
                    "auc": float(va["ROC_AUC"]) if np.isfinite(va["ROC_AUC"]) else np.nan,
                    "state": state,
                    "val_score01": val_score01,
                    "val_w": val_w,
                    "val_y": val_y,
                }
            )

            b = candidates[-1]["boyce"]
            b_key = -np.inf if not np.isfinite(b) else float(b)
            if b_key > best_boyce_key + 1e-9:
                best_boyce_key = b_key
                no_improve = 0
            else:
                no_improve += 1
        else:
            no_improve += 1

        if no_improve >= PATIENCE:
            print(f"[fold {fold_id}] early stopping at epoch {ep}")
            break

    if len(candidates) == 0:
        raise RuntimeError(
            f"[fold {fold_id}] No epoch met AUC floor (AUC_FLOOR={AUC_FLOOR}). "
            f"Lower AUC_FLOOR or set USE_AUC_FLOOR=False."
        )

    def key(c):
        b = c["boyce"]
        a = c["auc"]
        b_val = -np.inf if not np.isfinite(b) else float(b)
        a_val = -np.inf if not np.isfinite(a) else float(a)
        return (b_val, a_val)

    best = max(candidates, key=key)

    ckpt_path = os.path.join(model_dir, f"cnn_kulsif_fold_{fold_id}.pt")
    torch.save(best["state"], ckpt_path)
    print(
        f"[fold {fold_id}] selected epoch={best['epoch']} | "
        f"Boyce={best['boyce']:.4f}, AUC={best['auc']:.4f} | saved -> {ckpt_path}"
    )

    best_val = {"Boyce": best["boyce"], "ROC_AUC": best["auc"], "loss": np.nan}
    return best_val, best["val_score01"], val_idx


# =========================================================
# Ensemble prediction (KuLSIF)
# - each model outputs w
# - ensemble = weighted mean of w
# - bounded score01 = w/(1+w)
# =========================================================
def ensemble_predict_on_indices_kulsif_cnn(
    X_values: np.ndarray,
    X_masks: np.ndarray,
    y: np.ndarray,
    indices: np.ndarray,
    model_dir: str,
    fold_ids: List[int],
    weights: np.ndarray,
):
    in_value_channels = X_values.shape[1]

    models = []
    for fid in fold_ids:
        ckpt_path = os.path.join(model_dir, f"cnn_kulsif_fold_{fid}.pt")
        state = torch.load(ckpt_path, map_location="cpu")

        if state["in_value_channels"] != in_value_channels:
            raise RuntimeError(f"Value channel mismatch for fold {fid}")
        if state.get("patch_size", PATCH_SIZE) != PATCH_SIZE:
            raise RuntimeError(f"Patch size mismatch for fold {fid}")

        model = CNNKuLSIF(
            in_value_channels=in_value_channels,
            emb_dim=state.get("emb_dim", EMB_DIM),
            hidden_dims=tuple(state.get("hidden_dims", HIDDEN_DIMS)),
            dropout=DROPOUT,
            max_w=state.get("max_w", MAX_W),
            eps=state.get("w_eps", W_EPS),
        ).to(device)
        model.load_state_dict(state["model_state"])
        model.eval()
        models.append(model)

    w = np.asarray(weights, dtype=np.float64)
    if (not np.isfinite(w).all()) or w.sum() <= 0:
        w = np.ones(len(fold_ids), dtype=np.float64) / len(fold_ids)
    else:
        w = w / w.sum()

    weights_t = torch.tensor(w, dtype=torch.float32, device=device)

    idx = np.asarray(indices, dtype=np.int64)
    all_score01, all_w, all_y = [], [], []

    with torch.no_grad():
        for start in range(0, len(idx), BATCH_SIZE):
            end = min(start + BATCH_SIZE, len(idx))
            idx_batch = idx[start:end]

            xv = torch.from_numpy(X_values[idx_batch].astype(np.float32)).to(device)
            xm = torch.from_numpy(X_masks[idx_batch].astype(np.float32)).to(device)
            yb = torch.from_numpy(y[idx_batch].astype(np.float32)).to(device)

            w_ens = torch.zeros(xv.size(0), device=device)
            for k, model in enumerate(models):
                w_k = model(xv, xm)
                w_ens += weights_t[k] * w_k

            w_ens = torch.clamp(w_ens, 0.0, MAX_W)
            score01 = w_ens / (1.0 + w_ens)

            all_score01.append(score01.cpu().numpy())
            all_w.append(w_ens.cpu().numpy())
            all_y.append(yb.cpu().numpy())

    return np.concatenate(all_score01), np.concatenate(all_w), np.concatenate(all_y)


# =========================================================
# MAIN
# =========================================================
if __name__ == "__main__":
    set_seed(SEED)
    os.makedirs(MODEL_DIR, exist_ok=True)

    # ----------- load CV data -----------
    X_cv = np.load(CV_X_PATH)
    M_cv = np.load(CV_M_PATH)
    y_cv = np.load(CV_Y_PATH).astype(np.float32)
    fold_cv_raw = np.load(CV_FOLD_PATH)

    print("[CV] X shape:", X_cv.shape)
    print("[CV] M shape:", M_cv.shape)
    print("[CV] y shape:", y_cv.shape)
    print("[CV] fold shape:", fold_cv_raw.shape)

    if X_cv.shape[0] != y_cv.shape[0] or M_cv.shape[0] != y_cv.shape[0]:
        raise ValueError("Mismatch between CV X/M and y lengths")

    if not np.isfinite(fold_cv_raw).all():
        raise ValueError("fold.npy contains NaN/inf; CV split must be defined for all rows")
    fold_cv = fold_cv_raw.astype(int)

    bad = ~np.isfinite(X_cv)
    if bad.any():
        print(f"[CV] Warning: {int(bad.sum())} non-finite X values set to 0.")
        X_cv[bad] = 0.0
    bad_m = ~np.isfinite(M_cv)
    if bad_m.any():
        print(f"[CV] Warning: {int(bad_m.sum())} non-finite M values set to 0.")
        M_cv[bad_m] = 0.0

    # ----------- load TEST data -----------
    X_test = np.load(TEST_X_PATH)
    M_test = np.load(TEST_M_PATH)
    y_test = np.load(TEST_Y_PATH).astype(np.float32)

    print("[TEST] X shape:", X_test.shape)
    print("[TEST] M shape:", M_test.shape)
    print("[TEST] y shape:", y_test.shape)

    if X_test.shape[0] != y_test.shape[0] or M_test.shape[0] != y_test.shape[0]:
        raise ValueError("Mismatch between TEST X/M and y lengths")

    bad = ~np.isfinite(X_test)
    if bad.any():
        print(f"[TEST] Warning: {int(bad.sum())} non-finite X values set to 0.")
        X_test[bad] = 0.0
    bad_m = ~np.isfinite(M_test)
    if bad_m.any():
        print(f"[TEST] Warning: {int(bad_m.sum())} non-finite M values set to 0.")
        M_test[bad_m] = 0.0

    # ----------- CV training over folds -----------
    fold_ids = make_cv_indices(fold_cv)
    print("Folds:", fold_ids)

    # OOF bounded scores (0..1) for metrics comparability
    oof_score01 = np.full_like(y_cv, np.nan, dtype=float)

    fold_aucs = []
    fold_boyces = []

    for fid in fold_ids:
        best_val, val_score01, val_idx = train_and_save_fold_model_kulsif_cnn(
            fid, X_cv, M_cv, y_cv, fold_cv, MODEL_DIR
        )
        fold_aucs.append(best_val["ROC_AUC"])
        fold_boyces.append(best_val["Boyce"])
        oof_score01[val_idx] = val_score01

    # ----------- CV metrics (OOF bounded score) -----------
    cv_metrics = compute_boyce_and_auc(y_cv, oof_score01)
    print("\n[CV] OOF metrics (bounded score = w/(1+w)):", cv_metrics)
    print("[CV] per-fold best AUCs:", fold_aucs)
    print("[CV] per-fold best Boyce:", fold_boyces)

    # ----------- Ensemble weights -----------
    # KuLSIF doesn't give a comparable "loss" like BCE, so default to equal weights.
    weights = np.ones(len(fold_ids), dtype=float) / len(fold_ids)
    print("\n[Ensemble] Using equal weights:", weights)

    np.save(os.path.join(MODEL_DIR, "fold_ids.npy"), np.array(fold_ids, dtype=int))
    np.save(os.path.join(MODEL_DIR, "fold_weights_equal.npy"), weights)
    np.save(os.path.join(MODEL_DIR, "oof_score01.npy"), oof_score01)
    print("[Saved] fold_ids.npy, fold_weights_equal.npy, oof_score01.npy")

    # ----------- External test ensemble (bounded score) -----------
    test_idx = np.arange(len(y_test))
    print("External test N =", len(test_idx))

    test_score01, test_w, test_y = ensemble_predict_on_indices_kulsif_cnn(
        X_values=X_test,
        X_masks=M_test,
        y=y_test,
        indices=test_idx,
        model_dir=MODEL_DIR,
        fold_ids=fold_ids,
        weights=weights,
    )

    test_metrics = compute_boyce_and_auc(test_y, test_score01)
    print("\n[Test] Ensemble metrics (bounded score = w/(1+w)):", test_metrics)

    np.save(os.path.join(MODEL_DIR, "test_score01.npy"), test_score01)
    np.save(os.path.join(MODEL_DIR, "test_w_ens.npy"), test_w)
    print("[Test] Saved test_score01.npy, test_w_ens.npy")


# Patch size: 13


In [ ]:
import os
from typing import List, Dict, Tuple

import numpy as np

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW

from sklearn.metrics import roc_auc_score

# =========================================================
# CONFIG
# =========================================================

CV_DIR = str(DATA_ROOT / "cv_patches_13")
TEST_DIR = str(DATA_ROOT / "test_patches_13")

CV_X_PATH = os.path.join(CV_DIR, "X.npy")
CV_M_PATH = os.path.join(CV_DIR, "M.npy")
CV_Y_PATH = os.path.join(CV_DIR, "y.npy")
CV_FOLD_PATH = os.path.join(CV_DIR, "fold.npy")

TEST_X_PATH = os.path.join(TEST_DIR, "X.npy")
TEST_M_PATH = os.path.join(TEST_DIR, "M.npy")
TEST_Y_PATH = os.path.join(TEST_DIR, "y.npy")

MODEL_DIR = str(OUTPUT_ROOT / "kulsif" / "cnn_kulsif_patch_models_13")

BATCH_SIZE = 256
NUM_WORKERS = 0

MAX_EPOCHS = 100
PATIENCE = 10

LR = 1e-3
WEIGHT_DECAY = 1e-3

SEED = 42

EMB_DIM = 32
HIDDEN_DIMS = [32]
DROPOUT = 0.35

PATCH_SIZE = 13  # 13x13 patches

# Data augmentation
USE_FLIPS = True
NOISE_STD = 0.08

# Model selection gate (optional): computed on bounded monotone score01 = w/(1+w)
USE_AUC_FLOOR = False
AUC_FLOOR = 0.80

# Numeric stability
W_EPS = 1e-8
MAX_W = 1e6

# KuLSIF penalty strength (optional): encourages E_q[w] = 1
KULSIF_NORM_LAM = 10.0

# If your labels are reversed, flip these:
#   y==1 -> p (target)
#   y==0 -> q (reference)
P_LABEL = 1
Q_LABEL = 0


# =========================================================
# Device
# =========================================================
if torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


# =========================================================
# Metrics: Boyce + AUC (expects any real-valued score)
# =========================================================
def _average_ranks(x: np.ndarray) -> np.ndarray:
    x = np.asarray(x, dtype=float)
    order = np.argsort(x, kind="mergesort")
    ranks = np.empty_like(order, dtype=float)
    ranks[order] = np.arange(1, len(x) + 1, dtype=float)
    sx = x[order]
    i = 0
    while i < len(x):
        j = i + 1
        while j < len(x) and sx[j] == sx[i]:
            j += 1
        if j - i > 1:
            avg = (i + 1 + j) / 2.0
            ranks[order[i:j]] = avg
        i = j
    return ranks


def _spearman_tieaware(x, y) -> float:
    x = np.asarray(x, float)
    y = np.asarray(y, float)
    if x.size < 3 or y.size < 3:
        return np.nan
    rx, ry = _average_ranks(x), _average_ranks(y)
    if np.all(rx == rx[0]) or np.all(ry == ry[0]):
        return np.nan
    return float(np.corrcoef(rx, ry)[0, 1])


def continuous_boyce(y_true, scores, nbins_max=20, min_per_group=10):
    y = np.asarray(y_true).astype(int)
    s = np.asarray(scores, dtype=float)

    s_bg = s[y == 0]
    s_pr = s[y == 1]
    if (s_bg.size < min_per_group) or (s_pr.size < min_per_group):
        return np.nan

    uq = np.unique(s_bg[np.isfinite(s_bg)])
    if uq.size < 3:
        return np.nan
    nb = min(nbins_max, max(3, uq.size - 1))

    qs = np.quantile(s_bg, np.linspace(0.0, 1.0, nb + 1))
    qs[0] -= 1e-12
    qs[-1] += 1e-12

    pratio, centers = [], []
    Lb, Lp = float(len(s_bg)), float(len(s_pr))
    for a, b in zip(qs[:-1], qs[1:]):
        in_bg = (s_bg >= a) & (s_bg < b)
        nbk = int(in_bg.sum())
        if nbk == 0:
            continue
        in_pr = (s_pr >= a) & (s_pr < b)
        npk = int(in_pr.sum())
        pratio.append((npk / Lp) / (nbk / Lb))
        centers.append(0.5 * (a + b))

    if len(pratio) < 3:
        return np.nan
    return _spearman_tieaware(np.asarray(centers), np.asarray(pratio))


def compute_auc(y_true, scores):
    y = np.asarray(y_true, int)
    s = np.asarray(scores, float)
    m = np.isfinite(s)
    y, s = y[m], s[m]
    if y.size < 2 or np.unique(y).size < 2:
        return np.nan
    return float(roc_auc_score(y, s))


def compute_boyce_and_auc(y_true, scores, nbins_boyce=20):
    y = np.asarray(y_true, int)
    s = np.asarray(scores, float)
    m = np.isfinite(s)
    y, s = y[m], s[m]
    if y.size == 0:
        return dict(Boyce=np.nan, ROC_AUC=np.nan)
    return dict(
        Boyce=continuous_boyce(y, s, nbins_max=nbins_boyce),
        ROC_AUC=compute_auc(y, s),
    )


# =========================================================
# Helpers
# =========================================================
def w_to_score01_np(w: np.ndarray) -> np.ndarray:
    w = np.asarray(w, dtype=np.float64)
    w = np.clip(w, 0.0, MAX_W)
    return w / (1.0 + w)


def set_seed(seed=42):
    import random
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def make_cv_indices(fold_cv: np.ndarray) -> List[int]:
    vals = np.unique(fold_cv)
    return sorted(int(v) for v in vals)


# =========================================================
# Dataset + Models
# =========================================================
def augment_patch(values: torch.Tensor, mask: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
    if USE_FLIPS:
        if torch.rand(1).item() < 0.5:
            values = torch.flip(values, dims=[1])
            mask = torch.flip(mask, dims=[1])
        if torch.rand(1).item() < 0.5:
            values = torch.flip(values, dims=[2])
            mask = torch.flip(mask, dims=[2])

    if NOISE_STD > 0.0:
        noise = torch.randn_like(values) * NOISE_STD
        values = values + noise

    return values, mask


class PatchDREDataset(Dataset):
    """Returns xv, xm, y. Useful for validation metrics only."""
    def __init__(self, X_values, X_masks, y, indices, train: bool):
        super().__init__()
        self.Xv = X_values.astype(np.float32)
        self.Xm = X_masks.astype(np.float32)
        self.y = y.astype(np.float32)
        self.indices = np.asarray(indices, dtype=np.int64)
        self.train = train

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i: int):
        idx = self.indices[i]
        xv = torch.from_numpy(self.Xv[idx])
        xm = torch.from_numpy(self.Xm[idx])
        y = torch.tensor(self.y[idx], dtype=torch.float32)
        if self.train:
            xv, xm = augment_patch(xv, xm)
        return xv, xm, y


class PatchXDataset(Dataset):
    """Returns xv, xm only. Used for p/q loaders."""
    def __init__(self, X_values, X_masks, indices, train: bool):
        super().__init__()
        self.Xv = X_values.astype(np.float32)
        self.Xm = X_masks.astype(np.float32)
        self.indices = np.asarray(indices, dtype=np.int64)
        self.train = train

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i: int):
        idx = self.indices[i]
        xv = torch.from_numpy(self.Xv[idx])
        xm = torch.from_numpy(self.Xm[idx])
        if self.train:
            xv, xm = augment_patch(xv, xm)
        return xv, xm


class MLP(nn.Module):
    def __init__(self, in_dim: int, hidden_dims=(256, 128, 64), dropout=0.2):
        super().__init__()
        layers = []
        prev = in_dim
        for h in hidden_dims:
            layers.append(nn.Linear(prev, h))
            layers.append(nn.ReLU())
            if dropout > 0:
                layers.append(nn.Dropout(dropout))
            prev = h
        layers.append(nn.Linear(prev, 1))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x).squeeze(-1)


class PatchEncoder13(nn.Module):
    def __init__(self, in_value_channels: int, emb_dim: int = 32):
        super().__init__()
        in_channels = 2 * in_value_channels
        self.features = nn.Sequential(
            nn.Conv2d(in_channels, 16, 3, padding=1, bias=False),
            nn.BatchNorm2d(16),
            nn.ReLU(inplace=True),

            nn.Conv2d(16, 16, 3, padding=1, bias=False),
            nn.BatchNorm2d(16),
            nn.ReLU(inplace=True),

            nn.MaxPool2d(2, 2),  # 13→6

            nn.Conv2d(16, 32, 3, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),

            nn.AdaptiveAvgPool2d(1),
        )
        self.proj = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32, emb_dim),
            nn.ReLU(inplace=True),
        )

    def forward(self, x_val, x_mask):
        x_val = x_val * x_mask
        x_in = torch.cat([x_val, x_mask], dim=1)
        h = self.features(x_in)
        z = self.proj(h)
        return z


class CNNKuLSIF(nn.Module):
    """
    Neural KuLSIF/uLSIF: network outputs w(x) >= 0 (density ratio).
    Parameterization: w = softplus(raw) + eps.
    """
    def __init__(self, in_value_channels, emb_dim=64, hidden_dims=(128, 64), dropout=0.2,
                 max_w=1e6, eps=1e-8):
        super().__init__()
        self.encoder = PatchEncoder13(in_value_channels, emb_dim)
        self.head = MLP(in_dim=emb_dim, hidden_dims=hidden_dims, dropout=dropout)
        self.max_w = float(max_w) if max_w is not None else None
        self.eps = float(eps)

    def forward(self, x_val, x_mask):
        z = self.encoder(x_val, x_mask)
        raw = self.head(z)
        w = torch.nn.functional.softplus(raw) + self.eps
        if self.max_w is not None:
            w = torch.clamp(w, 0.0, self.max_w)
        return w


# =========================================================
# KuLSIF loss:
# minimize 0.5 E_q[w^2] - E_p[w]  (+ optional penalty for E_q[w]=1)
# =========================================================
def kulsif_loss(w_p: torch.Tensor, w_q: torch.Tensor, norm_lam: float) -> Tuple[torch.Tensor, float]:
    loss = 0.5 * (w_q ** 2).mean() - w_p.mean()
    norm_err = (w_q.mean() - 1.0)
    if norm_lam is not None and norm_lam > 0:
        loss = loss + norm_lam * (norm_err ** 2)
    return loss, float(norm_err.detach().cpu())


# =========================================================
# Training per fold (KuLSIF)
# - train on p/q batches
# - validation metrics computed on bounded score01 = w/(1+w)
# - returns OOF bounded score01 for that fold val split
# =========================================================
def train_and_save_fold_model_kulsif_cnn(
    fold_id: int,
    X_values: np.ndarray,
    X_masks: np.ndarray,
    y_cv: np.ndarray,
    fold_cv: np.ndarray,
    model_dir: str,
) -> Tuple[Dict[str, float], np.ndarray, np.ndarray]:
    os.makedirs(model_dir, exist_ok=True)

    f = fold_cv.astype(int)
    train_idx = np.where(f != fold_id)[0]
    val_idx = np.where(f == fold_id)[0]

    # p/q split inside train
    p_tr = train_idx[y_cv[train_idx] == P_LABEL]
    q_tr = train_idx[y_cv[train_idx] == Q_LABEL]
    p_va = val_idx[y_cv[val_idx] == P_LABEL]
    q_va = val_idx[y_cv[val_idx] == Q_LABEL]

    if len(p_tr) < 10 or len(q_tr) < 10:
        raise RuntimeError(f"[fold {fold_id}] too few samples: p_tr={len(p_tr)} q_tr={len(q_tr)}")

    print(f"\n[fold {fold_id}] train: p={len(p_tr)} q={len(q_tr)} | val: p={len(p_va)} q={len(q_va)}")

    ds_p_tr = PatchXDataset(X_values, X_masks, p_tr, train=True)
    ds_q_tr = PatchXDataset(X_values, X_masks, q_tr, train=True)
    ds_val = PatchDREDataset(X_values, X_masks, y_cv, val_idx, train=False)

    dl_p_tr = DataLoader(ds_p_tr, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, drop_last=True)
    dl_q_tr = DataLoader(ds_q_tr, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, drop_last=True)
    dl_val = DataLoader(ds_val, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, drop_last=False)

    in_value_channels = X_values.shape[1]
    model = CNNKuLSIF(
        in_value_channels=in_value_channels,
        emb_dim=EMB_DIM,
        hidden_dims=HIDDEN_DIMS,
        dropout=DROPOUT,
        max_w=MAX_W,
        eps=W_EPS,
    ).to(device)

    opt = AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

    def run_train_epoch():
        model.train()
        total_loss = 0.0
        total_steps = 0
        norm_err_avg = 0.0

        for (xvp, xmp), (xvq, xmq) in zip(dl_p_tr, dl_q_tr):
            xvp, xmp = xvp.to(device), xmp.to(device)
            xvq, xmq = xvq.to(device), xmq.to(device)

            opt.zero_grad(set_to_none=True)

            w_p = model(xvp, xmp)
            w_q = model(xvq, xmq)

            loss, ne = kulsif_loss(w_p, w_q, norm_lam=KULSIF_NORM_LAM)

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()

            total_loss += float(loss.detach().cpu())
            norm_err_avg += ne
            total_steps += 1

        return {
            "kulsif_loss": total_loss / max(1, total_steps),
            "norm_err": norm_err_avg / max(1, total_steps),
        }

    def run_val_metrics():
        model.eval()
        all_w, all_y = [], []
        with torch.no_grad():
            for xv, xm, yb in dl_val:
                xv, xm = xv.to(device), xm.to(device)
                w = model(xv, xm)
                all_w.append(w.detach().cpu().numpy())
                all_y.append(yb.numpy())

        w_np = np.concatenate(all_w)
        y_np = np.concatenate(all_y).astype(int)

        score01 = w_to_score01_np(w_np)
        mets = compute_boyce_and_auc(y_np, score01)
        return mets, score01, w_np, y_np

    candidates = []
    best_boyce_key = -np.inf
    no_improve = 0

    for ep in range(1, MAX_EPOCHS + 1):
        tr = run_train_epoch()
        va, val_score01, val_w, val_y = run_val_metrics()

        print(
            f"[fold {fold_id}] epoch {ep:03d} | "
            f"train loss {tr['kulsif_loss']:.4f}, norm_err {tr['norm_err']:.4f} | "
            f"val AUC {va['ROC_AUC']:.4f}, Boyce {va['Boyce']:.4f}"
        )

        auc_ok = True
        if USE_AUC_FLOOR:
            auc_ok = np.isfinite(va["ROC_AUC"]) and (va["ROC_AUC"] >= AUC_FLOOR)

        if auc_ok:
            state = {
                "model_state": {k: v.detach().cpu().clone() for k, v in model.state_dict().items()},
                "in_value_channels": in_value_channels,
                "patch_size": PATCH_SIZE,
                "emb_dim": EMB_DIM,
                "hidden_dims": HIDDEN_DIMS,
                "kulsif_norm_lam": float(KULSIF_NORM_LAM),
                "max_w": float(MAX_W),
                "w_eps": float(W_EPS),
                "p_label": int(P_LABEL),
                "q_label": int(Q_LABEL),
            }
            candidates.append(
                {
                    "epoch": ep,
                    "boyce": float(va["Boyce"]) if np.isfinite(va["Boyce"]) else np.nan,
                    "auc": float(va["ROC_AUC"]) if np.isfinite(va["ROC_AUC"]) else np.nan,
                    "state": state,
                    "val_score01": val_score01,
                    "val_w": val_w,
                    "val_y": val_y,
                }
            )

            b = candidates[-1]["boyce"]
            b_key = -np.inf if not np.isfinite(b) else float(b)
            if b_key > best_boyce_key + 1e-9:
                best_boyce_key = b_key
                no_improve = 0
            else:
                no_improve += 1
        else:
            no_improve += 1

        if no_improve >= PATIENCE:
            print(f"[fold {fold_id}] early stopping at epoch {ep}")
            break

    if len(candidates) == 0:
        raise RuntimeError(
            f"[fold {fold_id}] No epoch met AUC floor (AUC_FLOOR={AUC_FLOOR}). "
            f"Lower AUC_FLOOR or set USE_AUC_FLOOR=False."
        )

    def key(c):
        b = c["boyce"]
        a = c["auc"]
        b_val = -np.inf if not np.isfinite(b) else float(b)
        a_val = -np.inf if not np.isfinite(a) else float(a)
        return (b_val, a_val)

    best = max(candidates, key=key)

    ckpt_path = os.path.join(model_dir, f"cnn_kulsif_fold_{fold_id}.pt")
    torch.save(best["state"], ckpt_path)
    print(
        f"[fold {fold_id}] selected epoch={best['epoch']} | "
        f"Boyce={best['boyce']:.4f}, AUC={best['auc']:.4f} | saved -> {ckpt_path}"
    )

    best_val = {"Boyce": best["boyce"], "ROC_AUC": best["auc"], "loss": np.nan}
    return best_val, best["val_score01"], val_idx


# =========================================================
# Ensemble prediction (KuLSIF)
# - each model outputs w
# - ensemble = weighted mean of w
# - bounded score01 = w/(1+w)
# =========================================================
def ensemble_predict_on_indices_kulsif_cnn(
    X_values: np.ndarray,
    X_masks: np.ndarray,
    y: np.ndarray,
    indices: np.ndarray,
    model_dir: str,
    fold_ids: List[int],
    weights: np.ndarray,
):
    in_value_channels = X_values.shape[1]

    models = []
    for fid in fold_ids:
        ckpt_path = os.path.join(model_dir, f"cnn_kulsif_fold_{fid}.pt")
        state = torch.load(ckpt_path, map_location="cpu")

        if state["in_value_channels"] != in_value_channels:
            raise RuntimeError(f"Value channel mismatch for fold {fid}")
        if state.get("patch_size", PATCH_SIZE) != PATCH_SIZE:
            raise RuntimeError(f"Patch size mismatch for fold {fid}")

        model = CNNKuLSIF(
            in_value_channels=in_value_channels,
            emb_dim=state.get("emb_dim", EMB_DIM),
            hidden_dims=tuple(state.get("hidden_dims", HIDDEN_DIMS)),
            dropout=DROPOUT,
            max_w=state.get("max_w", MAX_W),
            eps=state.get("w_eps", W_EPS),
        ).to(device)
        model.load_state_dict(state["model_state"])
        model.eval()
        models.append(model)

    w = np.asarray(weights, dtype=np.float64)
    if (not np.isfinite(w).all()) or w.sum() <= 0:
        w = np.ones(len(fold_ids), dtype=np.float64) / len(fold_ids)
    else:
        w = w / w.sum()

    weights_t = torch.tensor(w, dtype=torch.float32, device=device)

    idx = np.asarray(indices, dtype=np.int64)
    all_score01, all_w, all_y = [], [], []

    with torch.no_grad():
        for start in range(0, len(idx), BATCH_SIZE):
            end = min(start + BATCH_SIZE, len(idx))
            idx_batch = idx[start:end]

            xv = torch.from_numpy(X_values[idx_batch].astype(np.float32)).to(device)
            xm = torch.from_numpy(X_masks[idx_batch].astype(np.float32)).to(device)
            yb = torch.from_numpy(y[idx_batch].astype(np.float32)).to(device)

            w_ens = torch.zeros(xv.size(0), device=device)
            for k, model in enumerate(models):
                w_k = model(xv, xm)
                w_ens += weights_t[k] * w_k

            w_ens = torch.clamp(w_ens, 0.0, MAX_W)
            score01 = w_ens / (1.0 + w_ens)

            all_score01.append(score01.cpu().numpy())
            all_w.append(w_ens.cpu().numpy())
            all_y.append(yb.cpu().numpy())

    return np.concatenate(all_score01), np.concatenate(all_w), np.concatenate(all_y)


# =========================================================
# MAIN
# =========================================================
if __name__ == "__main__":
    set_seed(SEED)
    os.makedirs(MODEL_DIR, exist_ok=True)

    # ----------- load CV data -----------
    X_cv = np.load(CV_X_PATH)
    M_cv = np.load(CV_M_PATH)
    y_cv = np.load(CV_Y_PATH).astype(np.float32)
    fold_cv_raw = np.load(CV_FOLD_PATH)

    print("[CV] X shape:", X_cv.shape)
    print("[CV] M shape:", M_cv.shape)
    print("[CV] y shape:", y_cv.shape)
    print("[CV] fold shape:", fold_cv_raw.shape)

    if X_cv.shape[0] != y_cv.shape[0] or M_cv.shape[0] != y_cv.shape[0]:
        raise ValueError("Mismatch between CV X/M and y lengths")

    if not np.isfinite(fold_cv_raw).all():
        raise ValueError("fold.npy contains NaN/inf; CV split must be defined for all rows")
    fold_cv = fold_cv_raw.astype(int)

    bad = ~np.isfinite(X_cv)
    if bad.any():
        print(f"[CV] Warning: {int(bad.sum())} non-finite X values set to 0.")
        X_cv[bad] = 0.0
    bad_m = ~np.isfinite(M_cv)
    if bad_m.any():
        print(f"[CV] Warning: {int(bad_m.sum())} non-finite M values set to 0.")
        M_cv[bad_m] = 0.0

    # ----------- load TEST data -----------
    X_test = np.load(TEST_X_PATH)
    M_test = np.load(TEST_M_PATH)
    y_test = np.load(TEST_Y_PATH).astype(np.float32)

    print("[TEST] X shape:", X_test.shape)
    print("[TEST] M shape:", M_test.shape)
    print("[TEST] y shape:", y_test.shape)

    if X_test.shape[0] != y_test.shape[0] or M_test.shape[0] != y_test.shape[0]:
        raise ValueError("Mismatch between TEST X/M and y lengths")

    bad = ~np.isfinite(X_test)
    if bad.any():
        print(f"[TEST] Warning: {int(bad.sum())} non-finite X values set to 0.")
        X_test[bad] = 0.0
    bad_m = ~np.isfinite(M_test)
    if bad_m.any():
        print(f"[TEST] Warning: {int(bad_m.sum())} non-finite M values set to 0.")
        M_test[bad_m] = 0.0

    # ----------- CV training over folds -----------
    fold_ids = make_cv_indices(fold_cv)
    print("Folds:", fold_ids)

    # OOF bounded scores (0..1) for metrics comparability
    oof_score01 = np.full_like(y_cv, np.nan, dtype=float)

    fold_aucs = []
    fold_boyces = []

    for fid in fold_ids:
        best_val, val_score01, val_idx = train_and_save_fold_model_kulsif_cnn(
            fid, X_cv, M_cv, y_cv, fold_cv, MODEL_DIR
        )
        fold_aucs.append(best_val["ROC_AUC"])
        fold_boyces.append(best_val["Boyce"])
        oof_score01[val_idx] = val_score01

    # ----------- CV metrics (OOF bounded score) -----------
    cv_metrics = compute_boyce_and_auc(y_cv, oof_score01)
    print("\n[CV] OOF metrics (bounded score = w/(1+w)):", cv_metrics)
    print("[CV] per-fold best AUCs:", fold_aucs)
    print("[CV] per-fold best Boyce:", fold_boyces)

    # ----------- Ensemble weights -----------
    # KuLSIF doesn't give a comparable "loss" like BCE, so default to equal weights.
    weights = np.ones(len(fold_ids), dtype=float) / len(fold_ids)
    print("\n[Ensemble] Using equal weights:", weights)

    np.save(os.path.join(MODEL_DIR, "fold_ids.npy"), np.array(fold_ids, dtype=int))
    np.save(os.path.join(MODEL_DIR, "fold_weights_equal.npy"), weights)
    np.save(os.path.join(MODEL_DIR, "oof_score01.npy"), oof_score01)
    print("[Saved] fold_ids.npy, fold_weights_equal.npy, oof_score01.npy")

    # ----------- External test ensemble (bounded score) -----------
    test_idx = np.arange(len(y_test))
    print("External test N =", len(test_idx))

    test_score01, test_w, test_y = ensemble_predict_on_indices_kulsif_cnn(
        X_values=X_test,
        X_masks=M_test,
        y=y_test,
        indices=test_idx,
        model_dir=MODEL_DIR,
        fold_ids=fold_ids,
        weights=weights,
    )

    test_metrics = compute_boyce_and_auc(test_y, test_score01)
    print("\n[Test] Ensemble metrics (bounded score = w/(1+w)):", test_metrics)

    np.save(os.path.join(MODEL_DIR, "test_score01.npy"), test_score01)
    np.save(os.path.join(MODEL_DIR, "test_w_ens.npy"), test_w)
    print("[Test] Saved test_score01.npy, test_w_ens.npy")


# patch size: 33


In [ ]:
import os
from typing import List, Dict, Tuple

import numpy as np

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW

from sklearn.metrics import roc_auc_score

# =========================================================
# CONFIG
# =========================================================

CV_DIR = str(DATA_ROOT / "cv_patches_33")
TEST_DIR = str(DATA_ROOT / "test_patches_33")

CV_X_PATH = os.path.join(CV_DIR, "X.npy")
CV_M_PATH = os.path.join(CV_DIR, "M.npy")
CV_Y_PATH = os.path.join(CV_DIR, "y.npy")
CV_FOLD_PATH = os.path.join(CV_DIR, "fold.npy")

TEST_X_PATH = os.path.join(TEST_DIR, "X.npy")
TEST_M_PATH = os.path.join(TEST_DIR, "M.npy")
TEST_Y_PATH = os.path.join(TEST_DIR, "y.npy")

MODEL_DIR = str(OUTPUT_ROOT / "kulsif" / "cnn_kulsif_patch_models_33")

BATCH_SIZE = 256
NUM_WORKERS = 0

MAX_EPOCHS = 100
PATIENCE = 10

LR = 1e-3
WEIGHT_DECAY = 1e-3

SEED = 42

EMB_DIM = 32
HIDDEN_DIMS = [32]
DROPOUT = 0.35

PATCH_SIZE = 33  # 13x13 patches

# Data augmentation
USE_FLIPS = True
NOISE_STD = 0.08

# Model selection gate (optional): computed on bounded monotone score01 = w/(1+w)
USE_AUC_FLOOR = False
AUC_FLOOR = 0.80

# Numeric stability
W_EPS = 1e-8
MAX_W = 1e6

# KuLSIF penalty strength (optional): encourages E_q[w] = 1
KULSIF_NORM_LAM = 10.0

# If your labels are reversed, flip these:
#   y==1 -> p (target)
#   y==0 -> q (reference)
P_LABEL = 1
Q_LABEL = 0


# =========================================================
# Device
# =========================================================
if torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


# =========================================================
# Metrics: Boyce + AUC (expects any real-valued score)
# =========================================================
def _average_ranks(x: np.ndarray) -> np.ndarray:
    x = np.asarray(x, dtype=float)
    order = np.argsort(x, kind="mergesort")
    ranks = np.empty_like(order, dtype=float)
    ranks[order] = np.arange(1, len(x) + 1, dtype=float)
    sx = x[order]
    i = 0
    while i < len(x):
        j = i + 1
        while j < len(x) and sx[j] == sx[i]:
            j += 1
        if j - i > 1:
            avg = (i + 1 + j) / 2.0
            ranks[order[i:j]] = avg
        i = j
    return ranks


def _spearman_tieaware(x, y) -> float:
    x = np.asarray(x, float)
    y = np.asarray(y, float)
    if x.size < 3 or y.size < 3:
        return np.nan
    rx, ry = _average_ranks(x), _average_ranks(y)
    if np.all(rx == rx[0]) or np.all(ry == ry[0]):
        return np.nan
    return float(np.corrcoef(rx, ry)[0, 1])


def continuous_boyce(y_true, scores, nbins_max=20, min_per_group=10):
    y = np.asarray(y_true).astype(int)
    s = np.asarray(scores, dtype=float)

    s_bg = s[y == 0]
    s_pr = s[y == 1]
    if (s_bg.size < min_per_group) or (s_pr.size < min_per_group):
        return np.nan

    uq = np.unique(s_bg[np.isfinite(s_bg)])
    if uq.size < 3:
        return np.nan
    nb = min(nbins_max, max(3, uq.size - 1))

    qs = np.quantile(s_bg, np.linspace(0.0, 1.0, nb + 1))
    qs[0] -= 1e-12
    qs[-1] += 1e-12

    pratio, centers = [], []
    Lb, Lp = float(len(s_bg)), float(len(s_pr))
    for a, b in zip(qs[:-1], qs[1:]):
        in_bg = (s_bg >= a) & (s_bg < b)
        nbk = int(in_bg.sum())
        if nbk == 0:
            continue
        in_pr = (s_pr >= a) & (s_pr < b)
        npk = int(in_pr.sum())
        pratio.append((npk / Lp) / (nbk / Lb))
        centers.append(0.5 * (a + b))

    if len(pratio) < 3:
        return np.nan
    return _spearman_tieaware(np.asarray(centers), np.asarray(pratio))


def compute_auc(y_true, scores):
    y = np.asarray(y_true, int)
    s = np.asarray(scores, float)
    m = np.isfinite(s)
    y, s = y[m], s[m]
    if y.size < 2 or np.unique(y).size < 2:
        return np.nan
    return float(roc_auc_score(y, s))


def compute_boyce_and_auc(y_true, scores, nbins_boyce=20):
    y = np.asarray(y_true, int)
    s = np.asarray(scores, float)
    m = np.isfinite(s)
    y, s = y[m], s[m]
    if y.size == 0:
        return dict(Boyce=np.nan, ROC_AUC=np.nan)
    return dict(
        Boyce=continuous_boyce(y, s, nbins_max=nbins_boyce),
        ROC_AUC=compute_auc(y, s),
    )


# =========================================================
# Helpers
# =========================================================
def w_to_score01_np(w: np.ndarray) -> np.ndarray:
    w = np.asarray(w, dtype=np.float64)
    w = np.clip(w, 0.0, MAX_W)
    return w / (1.0 + w)


def set_seed(seed=42):
    import random
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def make_cv_indices(fold_cv: np.ndarray) -> List[int]:
    vals = np.unique(fold_cv)
    return sorted(int(v) for v in vals)


# =========================================================
# Dataset + Models
# =========================================================
def augment_patch(values: torch.Tensor, mask: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
    if USE_FLIPS:
        if torch.rand(1).item() < 0.5:
            values = torch.flip(values, dims=[1])
            mask = torch.flip(mask, dims=[1])
        if torch.rand(1).item() < 0.5:
            values = torch.flip(values, dims=[2])
            mask = torch.flip(mask, dims=[2])

    if NOISE_STD > 0.0:
        noise = torch.randn_like(values) * NOISE_STD
        values = values + noise

    return values, mask


class PatchDREDataset(Dataset):
    """Returns xv, xm, y. Useful for validation metrics only."""
    def __init__(self, X_values, X_masks, y, indices, train: bool):
        super().__init__()
        self.Xv = X_values.astype(np.float32)
        self.Xm = X_masks.astype(np.float32)
        self.y = y.astype(np.float32)
        self.indices = np.asarray(indices, dtype=np.int64)
        self.train = train

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i: int):
        idx = self.indices[i]
        xv = torch.from_numpy(self.Xv[idx])
        xm = torch.from_numpy(self.Xm[idx])
        y = torch.tensor(self.y[idx], dtype=torch.float32)
        if self.train:
            xv, xm = augment_patch(xv, xm)
        return xv, xm, y


class PatchXDataset(Dataset):
    """Returns xv, xm only. Used for p/q loaders."""
    def __init__(self, X_values, X_masks, indices, train: bool):
        super().__init__()
        self.Xv = X_values.astype(np.float32)
        self.Xm = X_masks.astype(np.float32)
        self.indices = np.asarray(indices, dtype=np.int64)
        self.train = train

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i: int):
        idx = self.indices[i]
        xv = torch.from_numpy(self.Xv[idx])
        xm = torch.from_numpy(self.Xm[idx])
        if self.train:
            xv, xm = augment_patch(xv, xm)
        return xv, xm


class MLP(nn.Module):
    def __init__(self, in_dim: int, hidden_dims=(256, 128, 64), dropout=0.2):
        super().__init__()
        layers = []
        prev = in_dim
        for h in hidden_dims:
            layers.append(nn.Linear(prev, h))
            layers.append(nn.ReLU())
            if dropout > 0:
                layers.append(nn.Dropout(dropout))
            prev = h
        layers.append(nn.Linear(prev, 1))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x).squeeze(-1)


class PatchEncoder13(nn.Module):
    def __init__(self, in_value_channels: int, emb_dim: int = 32):
        super().__init__()
        in_channels = 2 * in_value_channels
        self.features = nn.Sequential(
            nn.Conv2d(in_channels, 16, 3, padding=1, bias=False),
            nn.BatchNorm2d(16),
            nn.ReLU(inplace=True),

            nn.Conv2d(16, 16, 3, padding=1, bias=False),
            nn.BatchNorm2d(16),
            nn.ReLU(inplace=True),

            nn.MaxPool2d(2, 2),  # 13→6

            nn.Conv2d(16, 32, 3, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),

            nn.AdaptiveAvgPool2d(1),
        )
        self.proj = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32, emb_dim),
            nn.ReLU(inplace=True),
        )

    def forward(self, x_val, x_mask):
        x_val = x_val * x_mask
        x_in = torch.cat([x_val, x_mask], dim=1)
        h = self.features(x_in)
        z = self.proj(h)
        return z


class CNNKuLSIF(nn.Module):
    """
    Neural KuLSIF/uLSIF: network outputs w(x) >= 0 (density ratio).
    Parameterization: w = softplus(raw) + eps.
    """
    def __init__(self, in_value_channels, emb_dim=64, hidden_dims=(128, 64), dropout=0.2,
                 max_w=1e6, eps=1e-8):
        super().__init__()
        self.encoder = PatchEncoder13(in_value_channels, emb_dim)
        self.head = MLP(in_dim=emb_dim, hidden_dims=hidden_dims, dropout=dropout)
        self.max_w = float(max_w) if max_w is not None else None
        self.eps = float(eps)

    def forward(self, x_val, x_mask):
        z = self.encoder(x_val, x_mask)
        raw = self.head(z)
        w = torch.nn.functional.softplus(raw) + self.eps
        if self.max_w is not None:
            w = torch.clamp(w, 0.0, self.max_w)
        return w


# =========================================================
# KuLSIF loss:
# minimize 0.5 E_q[w^2] - E_p[w]  (+ optional penalty for E_q[w]=1)
# =========================================================
def kulsif_loss(w_p: torch.Tensor, w_q: torch.Tensor, norm_lam: float) -> Tuple[torch.Tensor, float]:
    loss = 0.5 * (w_q ** 2).mean() - w_p.mean()
    norm_err = (w_q.mean() - 1.0)
    if norm_lam is not None and norm_lam > 0:
        loss = loss + norm_lam * (norm_err ** 2)
    return loss, float(norm_err.detach().cpu())


# =========================================================
# Training per fold (KuLSIF)
# - train on p/q batches
# - validation metrics computed on bounded score01 = w/(1+w)
# - returns OOF bounded score01 for that fold val split
# =========================================================
def train_and_save_fold_model_kulsif_cnn(
    fold_id: int,
    X_values: np.ndarray,
    X_masks: np.ndarray,
    y_cv: np.ndarray,
    fold_cv: np.ndarray,
    model_dir: str,
) -> Tuple[Dict[str, float], np.ndarray, np.ndarray]:
    os.makedirs(model_dir, exist_ok=True)

    f = fold_cv.astype(int)
    train_idx = np.where(f != fold_id)[0]
    val_idx = np.where(f == fold_id)[0]

    # p/q split inside train
    p_tr = train_idx[y_cv[train_idx] == P_LABEL]
    q_tr = train_idx[y_cv[train_idx] == Q_LABEL]
    p_va = val_idx[y_cv[val_idx] == P_LABEL]
    q_va = val_idx[y_cv[val_idx] == Q_LABEL]

    if len(p_tr) < 10 or len(q_tr) < 10:
        raise RuntimeError(f"[fold {fold_id}] too few samples: p_tr={len(p_tr)} q_tr={len(q_tr)}")

    print(f"\n[fold {fold_id}] train: p={len(p_tr)} q={len(q_tr)} | val: p={len(p_va)} q={len(q_va)}")

    ds_p_tr = PatchXDataset(X_values, X_masks, p_tr, train=True)
    ds_q_tr = PatchXDataset(X_values, X_masks, q_tr, train=True)
    ds_val = PatchDREDataset(X_values, X_masks, y_cv, val_idx, train=False)

    dl_p_tr = DataLoader(ds_p_tr, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, drop_last=True)
    dl_q_tr = DataLoader(ds_q_tr, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, drop_last=True)
    dl_val = DataLoader(ds_val, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, drop_last=False)

    in_value_channels = X_values.shape[1]
    model = CNNKuLSIF(
        in_value_channels=in_value_channels,
        emb_dim=EMB_DIM,
        hidden_dims=HIDDEN_DIMS,
        dropout=DROPOUT,
        max_w=MAX_W,
        eps=W_EPS,
    ).to(device)

    opt = AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

    def run_train_epoch():
        model.train()
        total_loss = 0.0
        total_steps = 0
        norm_err_avg = 0.0

        for (xvp, xmp), (xvq, xmq) in zip(dl_p_tr, dl_q_tr):
            xvp, xmp = xvp.to(device), xmp.to(device)
            xvq, xmq = xvq.to(device), xmq.to(device)

            opt.zero_grad(set_to_none=True)

            w_p = model(xvp, xmp)
            w_q = model(xvq, xmq)

            loss, ne = kulsif_loss(w_p, w_q, norm_lam=KULSIF_NORM_LAM)

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()

            total_loss += float(loss.detach().cpu())
            norm_err_avg += ne
            total_steps += 1

        return {
            "kulsif_loss": total_loss / max(1, total_steps),
            "norm_err": norm_err_avg / max(1, total_steps),
        }

    def run_val_metrics():
        model.eval()
        all_w, all_y = [], []
        with torch.no_grad():
            for xv, xm, yb in dl_val:
                xv, xm = xv.to(device), xm.to(device)
                w = model(xv, xm)
                all_w.append(w.detach().cpu().numpy())
                all_y.append(yb.numpy())

        w_np = np.concatenate(all_w)
        y_np = np.concatenate(all_y).astype(int)

        score01 = w_to_score01_np(w_np)
        mets = compute_boyce_and_auc(y_np, score01)
        return mets, score01, w_np, y_np

    candidates = []
    best_boyce_key = -np.inf
    no_improve = 0

    for ep in range(1, MAX_EPOCHS + 1):
        tr = run_train_epoch()
        va, val_score01, val_w, val_y = run_val_metrics()

        print(
            f"[fold {fold_id}] epoch {ep:03d} | "
            f"train loss {tr['kulsif_loss']:.4f}, norm_err {tr['norm_err']:.4f} | "
            f"val AUC {va['ROC_AUC']:.4f}, Boyce {va['Boyce']:.4f}"
        )

        auc_ok = True
        if USE_AUC_FLOOR:
            auc_ok = np.isfinite(va["ROC_AUC"]) and (va["ROC_AUC"] >= AUC_FLOOR)

        if auc_ok:
            state = {
                "model_state": {k: v.detach().cpu().clone() for k, v in model.state_dict().items()},
                "in_value_channels": in_value_channels,
                "patch_size": PATCH_SIZE,
                "emb_dim": EMB_DIM,
                "hidden_dims": HIDDEN_DIMS,
                "kulsif_norm_lam": float(KULSIF_NORM_LAM),
                "max_w": float(MAX_W),
                "w_eps": float(W_EPS),
                "p_label": int(P_LABEL),
                "q_label": int(Q_LABEL),
            }
            candidates.append(
                {
                    "epoch": ep,
                    "boyce": float(va["Boyce"]) if np.isfinite(va["Boyce"]) else np.nan,
                    "auc": float(va["ROC_AUC"]) if np.isfinite(va["ROC_AUC"]) else np.nan,
                    "state": state,
                    "val_score01": val_score01,
                    "val_w": val_w,
                    "val_y": val_y,
                }
            )

            b = candidates[-1]["boyce"]
            b_key = -np.inf if not np.isfinite(b) else float(b)
            if b_key > best_boyce_key + 1e-9:
                best_boyce_key = b_key
                no_improve = 0
            else:
                no_improve += 1
        else:
            no_improve += 1

        if no_improve >= PATIENCE:
            print(f"[fold {fold_id}] early stopping at epoch {ep}")
            break

    if len(candidates) == 0:
        raise RuntimeError(
            f"[fold {fold_id}] No epoch met AUC floor (AUC_FLOOR={AUC_FLOOR}). "
            f"Lower AUC_FLOOR or set USE_AUC_FLOOR=False."
        )

    def key(c):
        b = c["boyce"]
        a = c["auc"]
        b_val = -np.inf if not np.isfinite(b) else float(b)
        a_val = -np.inf if not np.isfinite(a) else float(a)
        return (b_val, a_val)

    best = max(candidates, key=key)

    ckpt_path = os.path.join(model_dir, f"cnn_kulsif_fold_{fold_id}.pt")
    torch.save(best["state"], ckpt_path)
    print(
        f"[fold {fold_id}] selected epoch={best['epoch']} | "
        f"Boyce={best['boyce']:.4f}, AUC={best['auc']:.4f} | saved -> {ckpt_path}"
    )

    best_val = {"Boyce": best["boyce"], "ROC_AUC": best["auc"], "loss": np.nan}
    return best_val, best["val_score01"], val_idx


# =========================================================
# Ensemble prediction (KuLSIF)
# - each model outputs w
# - ensemble = weighted mean of w
# - bounded score01 = w/(1+w)
# =========================================================
def ensemble_predict_on_indices_kulsif_cnn(
    X_values: np.ndarray,
    X_masks: np.ndarray,
    y: np.ndarray,
    indices: np.ndarray,
    model_dir: str,
    fold_ids: List[int],
    weights: np.ndarray,
):
    in_value_channels = X_values.shape[1]

    models = []
    for fid in fold_ids:
        ckpt_path = os.path.join(model_dir, f"cnn_kulsif_fold_{fid}.pt")
        state = torch.load(ckpt_path, map_location="cpu")

        if state["in_value_channels"] != in_value_channels:
            raise RuntimeError(f"Value channel mismatch for fold {fid}")
        if state.get("patch_size", PATCH_SIZE) != PATCH_SIZE:
            raise RuntimeError(f"Patch size mismatch for fold {fid}")

        model = CNNKuLSIF(
            in_value_channels=in_value_channels,
            emb_dim=state.get("emb_dim", EMB_DIM),
            hidden_dims=tuple(state.get("hidden_dims", HIDDEN_DIMS)),
            dropout=DROPOUT,
            max_w=state.get("max_w", MAX_W),
            eps=state.get("w_eps", W_EPS),
        ).to(device)
        model.load_state_dict(state["model_state"])
        model.eval()
        models.append(model)

    w = np.asarray(weights, dtype=np.float64)
    if (not np.isfinite(w).all()) or w.sum() <= 0:
        w = np.ones(len(fold_ids), dtype=np.float64) / len(fold_ids)
    else:
        w = w / w.sum()

    weights_t = torch.tensor(w, dtype=torch.float32, device=device)

    idx = np.asarray(indices, dtype=np.int64)
    all_score01, all_w, all_y = [], [], []

    with torch.no_grad():
        for start in range(0, len(idx), BATCH_SIZE):
            end = min(start + BATCH_SIZE, len(idx))
            idx_batch = idx[start:end]

            xv = torch.from_numpy(X_values[idx_batch].astype(np.float32)).to(device)
            xm = torch.from_numpy(X_masks[idx_batch].astype(np.float32)).to(device)
            yb = torch.from_numpy(y[idx_batch].astype(np.float32)).to(device)

            w_ens = torch.zeros(xv.size(0), device=device)
            for k, model in enumerate(models):
                w_k = model(xv, xm)
                w_ens += weights_t[k] * w_k

            w_ens = torch.clamp(w_ens, 0.0, MAX_W)
            score01 = w_ens / (1.0 + w_ens)

            all_score01.append(score01.cpu().numpy())
            all_w.append(w_ens.cpu().numpy())
            all_y.append(yb.cpu().numpy())

    return np.concatenate(all_score01), np.concatenate(all_w), np.concatenate(all_y)


# =========================================================
# MAIN
# =========================================================
if __name__ == "__main__":
    set_seed(SEED)
    os.makedirs(MODEL_DIR, exist_ok=True)

    # ----------- load CV data -----------
    X_cv = np.load(CV_X_PATH)
    M_cv = np.load(CV_M_PATH)
    y_cv = np.load(CV_Y_PATH).astype(np.float32)
    fold_cv_raw = np.load(CV_FOLD_PATH)

    print("[CV] X shape:", X_cv.shape)
    print("[CV] M shape:", M_cv.shape)
    print("[CV] y shape:", y_cv.shape)
    print("[CV] fold shape:", fold_cv_raw.shape)

    if X_cv.shape[0] != y_cv.shape[0] or M_cv.shape[0] != y_cv.shape[0]:
        raise ValueError("Mismatch between CV X/M and y lengths")

    if not np.isfinite(fold_cv_raw).all():
        raise ValueError("fold.npy contains NaN/inf; CV split must be defined for all rows")
    fold_cv = fold_cv_raw.astype(int)

    bad = ~np.isfinite(X_cv)
    if bad.any():
        print(f"[CV] Warning: {int(bad.sum())} non-finite X values set to 0.")
        X_cv[bad] = 0.0
    bad_m = ~np.isfinite(M_cv)
    if bad_m.any():
        print(f"[CV] Warning: {int(bad_m.sum())} non-finite M values set to 0.")
        M_cv[bad_m] = 0.0

    # ----------- load TEST data -----------
    X_test = np.load(TEST_X_PATH)
    M_test = np.load(TEST_M_PATH)
    y_test = np.load(TEST_Y_PATH).astype(np.float32)

    print("[TEST] X shape:", X_test.shape)
    print("[TEST] M shape:", M_test.shape)
    print("[TEST] y shape:", y_test.shape)

    if X_test.shape[0] != y_test.shape[0] or M_test.shape[0] != y_test.shape[0]:
        raise ValueError("Mismatch between TEST X/M and y lengths")

    bad = ~np.isfinite(X_test)
    if bad.any():
        print(f"[TEST] Warning: {int(bad.sum())} non-finite X values set to 0.")
        X_test[bad] = 0.0
    bad_m = ~np.isfinite(M_test)
    if bad_m.any():
        print(f"[TEST] Warning: {int(bad_m.sum())} non-finite M values set to 0.")
        M_test[bad_m] = 0.0

    # ----------- CV training over folds -----------
    fold_ids = make_cv_indices(fold_cv)
    print("Folds:", fold_ids)

    # OOF bounded scores (0..1) for metrics comparability
    oof_score01 = np.full_like(y_cv, np.nan, dtype=float)

    fold_aucs = []
    fold_boyces = []

    for fid in fold_ids:
        best_val, val_score01, val_idx = train_and_save_fold_model_kulsif_cnn(
            fid, X_cv, M_cv, y_cv, fold_cv, MODEL_DIR
        )
        fold_aucs.append(best_val["ROC_AUC"])
        fold_boyces.append(best_val["Boyce"])
        oof_score01[val_idx] = val_score01

    # ----------- CV metrics (OOF bounded score) -----------
    cv_metrics = compute_boyce_and_auc(y_cv, oof_score01)
    print("\n[CV] OOF metrics (bounded score = w/(1+w)):", cv_metrics)
    print("[CV] per-fold best AUCs:", fold_aucs)
    print("[CV] per-fold best Boyce:", fold_boyces)

    # ----------- Ensemble weights -----------
    # KuLSIF doesn't give a comparable "loss" like BCE, so default to equal weights.
    weights = np.ones(len(fold_ids), dtype=float) / len(fold_ids)
    print("\n[Ensemble] Using equal weights:", weights)

    np.save(os.path.join(MODEL_DIR, "fold_ids.npy"), np.array(fold_ids, dtype=int))
    np.save(os.path.join(MODEL_DIR, "fold_weights_equal.npy"), weights)
    np.save(os.path.join(MODEL_DIR, "oof_score01.npy"), oof_score01)
    print("[Saved] fold_ids.npy, fold_weights_equal.npy, oof_score01.npy")

    # ----------- External test ensemble (bounded score) -----------
    test_idx = np.arange(len(y_test))
    print("External test N =", len(test_idx))

    test_score01, test_w, test_y = ensemble_predict_on_indices_kulsif_cnn(
        X_values=X_test,
        X_masks=M_test,
        y=y_test,
        indices=test_idx,
        model_dir=MODEL_DIR,
        fold_ids=fold_ids,
        weights=weights,
    )

    test_metrics = compute_boyce_and_auc(test_y, test_score01)
    print("\n[Test] Ensemble metrics (bounded score = w/(1+w)):", test_metrics)

    np.save(os.path.join(MODEL_DIR, "test_score01.npy"), test_score01)
    np.save(os.path.join(MODEL_DIR, "test_w_ens.npy"), test_w)
    print("[Test] Saved test_score01.npy, test_w_ens.npy")


# Patch size: 65


In [ ]:
import os
from typing import List, Dict, Tuple

import numpy as np

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW

from sklearn.metrics import roc_auc_score

# =========================================================
# CONFIG
# =========================================================

CV_DIR = str(DATA_ROOT / "cv_patches_65")
TEST_DIR = str(DATA_ROOT / "test_patches_65")

CV_X_PATH = os.path.join(CV_DIR, "X.npy")
CV_M_PATH = os.path.join(CV_DIR, "M.npy")
CV_Y_PATH = os.path.join(CV_DIR, "y.npy")
CV_FOLD_PATH = os.path.join(CV_DIR, "fold.npy")

TEST_X_PATH = os.path.join(TEST_DIR, "X.npy")
TEST_M_PATH = os.path.join(TEST_DIR, "M.npy")
TEST_Y_PATH = os.path.join(TEST_DIR, "y.npy")

MODEL_DIR = str(OUTPUT_ROOT / "kulsif" / "cnn_kulsif_patch_models_65")

BATCH_SIZE = 256
NUM_WORKERS = 0

MAX_EPOCHS = 100
PATIENCE = 10

LR = 1e-3
WEIGHT_DECAY = 1e-3

SEED = 42

EMB_DIM = 32
HIDDEN_DIMS = [32]
DROPOUT = 0.35

PATCH_SIZE = 65  # 13x13 patches

# Data augmentation
USE_FLIPS = True
NOISE_STD = 0.08

# Model selection gate (optional): computed on bounded monotone score01 = w/(1+w)
USE_AUC_FLOOR = False
AUC_FLOOR = 0.80

# Numeric stability
W_EPS = 1e-8
MAX_W = 1e6

# KuLSIF penalty strength (optional): encourages E_q[w] = 1
KULSIF_NORM_LAM = 10.0

# If your labels are reversed, flip these:
#   y==1 -> p (target)
#   y==0 -> q (reference)
P_LABEL = 1
Q_LABEL = 0


# =========================================================
# Device
# =========================================================
if torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


# =========================================================
# Metrics: Boyce + AUC (expects any real-valued score)
# =========================================================
def _average_ranks(x: np.ndarray) -> np.ndarray:
    x = np.asarray(x, dtype=float)
    order = np.argsort(x, kind="mergesort")
    ranks = np.empty_like(order, dtype=float)
    ranks[order] = np.arange(1, len(x) + 1, dtype=float)
    sx = x[order]
    i = 0
    while i < len(x):
        j = i + 1
        while j < len(x) and sx[j] == sx[i]:
            j += 1
        if j - i > 1:
            avg = (i + 1 + j) / 2.0
            ranks[order[i:j]] = avg
        i = j
    return ranks


def _spearman_tieaware(x, y) -> float:
    x = np.asarray(x, float)
    y = np.asarray(y, float)
    if x.size < 3 or y.size < 3:
        return np.nan
    rx, ry = _average_ranks(x), _average_ranks(y)
    if np.all(rx == rx[0]) or np.all(ry == ry[0]):
        return np.nan
    return float(np.corrcoef(rx, ry)[0, 1])


def continuous_boyce(y_true, scores, nbins_max=20, min_per_group=10):
    y = np.asarray(y_true).astype(int)
    s = np.asarray(scores, dtype=float)

    s_bg = s[y == 0]
    s_pr = s[y == 1]
    if (s_bg.size < min_per_group) or (s_pr.size < min_per_group):
        return np.nan

    uq = np.unique(s_bg[np.isfinite(s_bg)])
    if uq.size < 3:
        return np.nan
    nb = min(nbins_max, max(3, uq.size - 1))

    qs = np.quantile(s_bg, np.linspace(0.0, 1.0, nb + 1))
    qs[0] -= 1e-12
    qs[-1] += 1e-12

    pratio, centers = [], []
    Lb, Lp = float(len(s_bg)), float(len(s_pr))
    for a, b in zip(qs[:-1], qs[1:]):
        in_bg = (s_bg >= a) & (s_bg < b)
        nbk = int(in_bg.sum())
        if nbk == 0:
            continue
        in_pr = (s_pr >= a) & (s_pr < b)
        npk = int(in_pr.sum())
        pratio.append((npk / Lp) / (nbk / Lb))
        centers.append(0.5 * (a + b))

    if len(pratio) < 3:
        return np.nan
    return _spearman_tieaware(np.asarray(centers), np.asarray(pratio))


def compute_auc(y_true, scores):
    y = np.asarray(y_true, int)
    s = np.asarray(scores, float)
    m = np.isfinite(s)
    y, s = y[m], s[m]
    if y.size < 2 or np.unique(y).size < 2:
        return np.nan
    return float(roc_auc_score(y, s))


def compute_boyce_and_auc(y_true, scores, nbins_boyce=20):
    y = np.asarray(y_true, int)
    s = np.asarray(scores, float)
    m = np.isfinite(s)
    y, s = y[m], s[m]
    if y.size == 0:
        return dict(Boyce=np.nan, ROC_AUC=np.nan)
    return dict(
        Boyce=continuous_boyce(y, s, nbins_max=nbins_boyce),
        ROC_AUC=compute_auc(y, s),
    )


# =========================================================
# Helpers
# =========================================================
def w_to_score01_np(w: np.ndarray) -> np.ndarray:
    w = np.asarray(w, dtype=np.float64)
    w = np.clip(w, 0.0, MAX_W)
    return w / (1.0 + w)


def set_seed(seed=42):
    import random
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def make_cv_indices(fold_cv: np.ndarray) -> List[int]:
    vals = np.unique(fold_cv)
    return sorted(int(v) for v in vals)


# =========================================================
# Dataset + Models
# =========================================================
def augment_patch(values: torch.Tensor, mask: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
    if USE_FLIPS:
        if torch.rand(1).item() < 0.5:
            values = torch.flip(values, dims=[1])
            mask = torch.flip(mask, dims=[1])
        if torch.rand(1).item() < 0.5:
            values = torch.flip(values, dims=[2])
            mask = torch.flip(mask, dims=[2])

    if NOISE_STD > 0.0:
        noise = torch.randn_like(values) * NOISE_STD
        values = values + noise

    return values, mask


class PatchDREDataset(Dataset):
    """Returns xv, xm, y. Useful for validation metrics only."""
    def __init__(self, X_values, X_masks, y, indices, train: bool):
        super().__init__()
        self.Xv = X_values.astype(np.float32)
        self.Xm = X_masks.astype(np.float32)
        self.y = y.astype(np.float32)
        self.indices = np.asarray(indices, dtype=np.int64)
        self.train = train

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i: int):
        idx = self.indices[i]
        xv = torch.from_numpy(self.Xv[idx])
        xm = torch.from_numpy(self.Xm[idx])
        y = torch.tensor(self.y[idx], dtype=torch.float32)
        if self.train:
            xv, xm = augment_patch(xv, xm)
        return xv, xm, y


class PatchXDataset(Dataset):
    """Returns xv, xm only. Used for p/q loaders."""
    def __init__(self, X_values, X_masks, indices, train: bool):
        super().__init__()
        self.Xv = X_values.astype(np.float32)
        self.Xm = X_masks.astype(np.float32)
        self.indices = np.asarray(indices, dtype=np.int64)
        self.train = train

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i: int):
        idx = self.indices[i]
        xv = torch.from_numpy(self.Xv[idx])
        xm = torch.from_numpy(self.Xm[idx])
        if self.train:
            xv, xm = augment_patch(xv, xm)
        return xv, xm


class MLP(nn.Module):
    def __init__(self, in_dim: int, hidden_dims=(256, 128, 64), dropout=0.2):
        super().__init__()
        layers = []
        prev = in_dim
        for h in hidden_dims:
            layers.append(nn.Linear(prev, h))
            layers.append(nn.ReLU())
            if dropout > 0:
                layers.append(nn.Dropout(dropout))
            prev = h
        layers.append(nn.Linear(prev, 1))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x).squeeze(-1)


class PatchEncoder13(nn.Module):
    def __init__(self, in_value_channels: int, emb_dim: int = 32):
        super().__init__()
        in_channels = 2 * in_value_channels
        self.features = nn.Sequential(
            nn.Conv2d(in_channels, 16, 3, padding=1, bias=False),
            nn.BatchNorm2d(16),
            nn.ReLU(inplace=True),

            nn.Conv2d(16, 16, 3, padding=1, bias=False),
            nn.BatchNorm2d(16),
            nn.ReLU(inplace=True),

            nn.MaxPool2d(2, 2),  # 13→6

            nn.Conv2d(16, 32, 3, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),

            nn.AdaptiveAvgPool2d(1),
        )
        self.proj = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32, emb_dim),
            nn.ReLU(inplace=True),
        )

    def forward(self, x_val, x_mask):
        x_val = x_val * x_mask
        x_in = torch.cat([x_val, x_mask], dim=1)
        h = self.features(x_in)
        z = self.proj(h)
        return z


class CNNKuLSIF(nn.Module):
    """
    Neural KuLSIF/uLSIF: network outputs w(x) >= 0 (density ratio).
    Parameterization: w = softplus(raw) + eps.
    """
    def __init__(self, in_value_channels, emb_dim=64, hidden_dims=(128, 64), dropout=0.2,
                 max_w=1e6, eps=1e-8):
        super().__init__()
        self.encoder = PatchEncoder13(in_value_channels, emb_dim)
        self.head = MLP(in_dim=emb_dim, hidden_dims=hidden_dims, dropout=dropout)
        self.max_w = float(max_w) if max_w is not None else None
        self.eps = float(eps)

    def forward(self, x_val, x_mask):
        z = self.encoder(x_val, x_mask)
        raw = self.head(z)
        w = torch.nn.functional.softplus(raw) + self.eps
        if self.max_w is not None:
            w = torch.clamp(w, 0.0, self.max_w)
        return w


# =========================================================
# KuLSIF loss:
# minimize 0.5 E_q[w^2] - E_p[w]  (+ optional penalty for E_q[w]=1)
# =========================================================
def kulsif_loss(w_p: torch.Tensor, w_q: torch.Tensor, norm_lam: float) -> Tuple[torch.Tensor, float]:
    loss = 0.5 * (w_q ** 2).mean() - w_p.mean()
    norm_err = (w_q.mean() - 1.0)
    if norm_lam is not None and norm_lam > 0:
        loss = loss + norm_lam * (norm_err ** 2)
    return loss, float(norm_err.detach().cpu())


# =========================================================
# Training per fold (KuLSIF)
# - train on p/q batches
# - validation metrics computed on bounded score01 = w/(1+w)
# - returns OOF bounded score01 for that fold val split
# =========================================================
def train_and_save_fold_model_kulsif_cnn(
    fold_id: int,
    X_values: np.ndarray,
    X_masks: np.ndarray,
    y_cv: np.ndarray,
    fold_cv: np.ndarray,
    model_dir: str,
) -> Tuple[Dict[str, float], np.ndarray, np.ndarray]:
    os.makedirs(model_dir, exist_ok=True)

    f = fold_cv.astype(int)
    train_idx = np.where(f != fold_id)[0]
    val_idx = np.where(f == fold_id)[0]

    # p/q split inside train
    p_tr = train_idx[y_cv[train_idx] == P_LABEL]
    q_tr = train_idx[y_cv[train_idx] == Q_LABEL]
    p_va = val_idx[y_cv[val_idx] == P_LABEL]
    q_va = val_idx[y_cv[val_idx] == Q_LABEL]

    if len(p_tr) < 10 or len(q_tr) < 10:
        raise RuntimeError(f"[fold {fold_id}] too few samples: p_tr={len(p_tr)} q_tr={len(q_tr)}")

    print(f"\n[fold {fold_id}] train: p={len(p_tr)} q={len(q_tr)} | val: p={len(p_va)} q={len(q_va)}")

    ds_p_tr = PatchXDataset(X_values, X_masks, p_tr, train=True)
    ds_q_tr = PatchXDataset(X_values, X_masks, q_tr, train=True)
    ds_val = PatchDREDataset(X_values, X_masks, y_cv, val_idx, train=False)

    dl_p_tr = DataLoader(ds_p_tr, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, drop_last=True)
    dl_q_tr = DataLoader(ds_q_tr, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, drop_last=True)
    dl_val = DataLoader(ds_val, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, drop_last=False)

    in_value_channels = X_values.shape[1]
    model = CNNKuLSIF(
        in_value_channels=in_value_channels,
        emb_dim=EMB_DIM,
        hidden_dims=HIDDEN_DIMS,
        dropout=DROPOUT,
        max_w=MAX_W,
        eps=W_EPS,
    ).to(device)

    opt = AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

    def run_train_epoch():
        model.train()
        total_loss = 0.0
        total_steps = 0
        norm_err_avg = 0.0

        for (xvp, xmp), (xvq, xmq) in zip(dl_p_tr, dl_q_tr):
            xvp, xmp = xvp.to(device), xmp.to(device)
            xvq, xmq = xvq.to(device), xmq.to(device)

            opt.zero_grad(set_to_none=True)

            w_p = model(xvp, xmp)
            w_q = model(xvq, xmq)

            loss, ne = kulsif_loss(w_p, w_q, norm_lam=KULSIF_NORM_LAM)

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()

            total_loss += float(loss.detach().cpu())
            norm_err_avg += ne
            total_steps += 1

        return {
            "kulsif_loss": total_loss / max(1, total_steps),
            "norm_err": norm_err_avg / max(1, total_steps),
        }

    def run_val_metrics():
        model.eval()
        all_w, all_y = [], []
        with torch.no_grad():
            for xv, xm, yb in dl_val:
                xv, xm = xv.to(device), xm.to(device)
                w = model(xv, xm)
                all_w.append(w.detach().cpu().numpy())
                all_y.append(yb.numpy())

        w_np = np.concatenate(all_w)
        y_np = np.concatenate(all_y).astype(int)

        score01 = w_to_score01_np(w_np)
        mets = compute_boyce_and_auc(y_np, score01)
        return mets, score01, w_np, y_np

    candidates = []
    best_boyce_key = -np.inf
    no_improve = 0

    for ep in range(1, MAX_EPOCHS + 1):
        tr = run_train_epoch()
        va, val_score01, val_w, val_y = run_val_metrics()

        print(
            f"[fold {fold_id}] epoch {ep:03d} | "
            f"train loss {tr['kulsif_loss']:.4f}, norm_err {tr['norm_err']:.4f} | "
            f"val AUC {va['ROC_AUC']:.4f}, Boyce {va['Boyce']:.4f}"
        )

        auc_ok = True
        if USE_AUC_FLOOR:
            auc_ok = np.isfinite(va["ROC_AUC"]) and (va["ROC_AUC"] >= AUC_FLOOR)

        if auc_ok:
            state = {
                "model_state": {k: v.detach().cpu().clone() for k, v in model.state_dict().items()},
                "in_value_channels": in_value_channels,
                "patch_size": PATCH_SIZE,
                "emb_dim": EMB_DIM,
                "hidden_dims": HIDDEN_DIMS,
                "kulsif_norm_lam": float(KULSIF_NORM_LAM),
                "max_w": float(MAX_W),
                "w_eps": float(W_EPS),
                "p_label": int(P_LABEL),
                "q_label": int(Q_LABEL),
            }
            candidates.append(
                {
                    "epoch": ep,
                    "boyce": float(va["Boyce"]) if np.isfinite(va["Boyce"]) else np.nan,
                    "auc": float(va["ROC_AUC"]) if np.isfinite(va["ROC_AUC"]) else np.nan,
                    "state": state,
                    "val_score01": val_score01,
                    "val_w": val_w,
                    "val_y": val_y,
                }
            )

            b = candidates[-1]["boyce"]
            b_key = -np.inf if not np.isfinite(b) else float(b)
            if b_key > best_boyce_key + 1e-9:
                best_boyce_key = b_key
                no_improve = 0
            else:
                no_improve += 1
        else:
            no_improve += 1

        if no_improve >= PATIENCE:
            print(f"[fold {fold_id}] early stopping at epoch {ep}")
            break

    if len(candidates) == 0:
        raise RuntimeError(
            f"[fold {fold_id}] No epoch met AUC floor (AUC_FLOOR={AUC_FLOOR}). "
            f"Lower AUC_FLOOR or set USE_AUC_FLOOR=False."
        )

    def key(c):
        b = c["boyce"]
        a = c["auc"]
        b_val = -np.inf if not np.isfinite(b) else float(b)
        a_val = -np.inf if not np.isfinite(a) else float(a)
        return (b_val, a_val)

    best = max(candidates, key=key)

    ckpt_path = os.path.join(model_dir, f"cnn_kulsif_fold_{fold_id}.pt")
    torch.save(best["state"], ckpt_path)
    print(
        f"[fold {fold_id}] selected epoch={best['epoch']} | "
        f"Boyce={best['boyce']:.4f}, AUC={best['auc']:.4f} | saved -> {ckpt_path}"
    )

    best_val = {"Boyce": best["boyce"], "ROC_AUC": best["auc"], "loss": np.nan}
    return best_val, best["val_score01"], val_idx


# =========================================================
# Ensemble prediction (KuLSIF)
# - each model outputs w
# - ensemble = weighted mean of w
# - bounded score01 = w/(1+w)
# =========================================================
def ensemble_predict_on_indices_kulsif_cnn(
    X_values: np.ndarray,
    X_masks: np.ndarray,
    y: np.ndarray,
    indices: np.ndarray,
    model_dir: str,
    fold_ids: List[int],
    weights: np.ndarray,
):
    in_value_channels = X_values.shape[1]

    models = []
    for fid in fold_ids:
        ckpt_path = os.path.join(model_dir, f"cnn_kulsif_fold_{fid}.pt")
        state = torch.load(ckpt_path, map_location="cpu")

        if state["in_value_channels"] != in_value_channels:
            raise RuntimeError(f"Value channel mismatch for fold {fid}")
        if state.get("patch_size", PATCH_SIZE) != PATCH_SIZE:
            raise RuntimeError(f"Patch size mismatch for fold {fid}")

        model = CNNKuLSIF(
            in_value_channels=in_value_channels,
            emb_dim=state.get("emb_dim", EMB_DIM),
            hidden_dims=tuple(state.get("hidden_dims", HIDDEN_DIMS)),
            dropout=DROPOUT,
            max_w=state.get("max_w", MAX_W),
            eps=state.get("w_eps", W_EPS),
        ).to(device)
        model.load_state_dict(state["model_state"])
        model.eval()
        models.append(model)

    w = np.asarray(weights, dtype=np.float64)
    if (not np.isfinite(w).all()) or w.sum() <= 0:
        w = np.ones(len(fold_ids), dtype=np.float64) / len(fold_ids)
    else:
        w = w / w.sum()

    weights_t = torch.tensor(w, dtype=torch.float32, device=device)

    idx = np.asarray(indices, dtype=np.int64)
    all_score01, all_w, all_y = [], [], []

    with torch.no_grad():
        for start in range(0, len(idx), BATCH_SIZE):
            end = min(start + BATCH_SIZE, len(idx))
            idx_batch = idx[start:end]

            xv = torch.from_numpy(X_values[idx_batch].astype(np.float32)).to(device)
            xm = torch.from_numpy(X_masks[idx_batch].astype(np.float32)).to(device)
            yb = torch.from_numpy(y[idx_batch].astype(np.float32)).to(device)

            w_ens = torch.zeros(xv.size(0), device=device)
            for k, model in enumerate(models):
                w_k = model(xv, xm)
                w_ens += weights_t[k] * w_k

            w_ens = torch.clamp(w_ens, 0.0, MAX_W)
            score01 = w_ens / (1.0 + w_ens)

            all_score01.append(score01.cpu().numpy())
            all_w.append(w_ens.cpu().numpy())
            all_y.append(yb.cpu().numpy())

    return np.concatenate(all_score01), np.concatenate(all_w), np.concatenate(all_y)


# =========================================================
# MAIN
# =========================================================
if __name__ == "__main__":
    set_seed(SEED)
    os.makedirs(MODEL_DIR, exist_ok=True)

    # ----------- load CV data -----------
    X_cv = np.load(CV_X_PATH)
    M_cv = np.load(CV_M_PATH)
    y_cv = np.load(CV_Y_PATH).astype(np.float32)
    fold_cv_raw = np.load(CV_FOLD_PATH)

    print("[CV] X shape:", X_cv.shape)
    print("[CV] M shape:", M_cv.shape)
    print("[CV] y shape:", y_cv.shape)
    print("[CV] fold shape:", fold_cv_raw.shape)

    if X_cv.shape[0] != y_cv.shape[0] or M_cv.shape[0] != y_cv.shape[0]:
        raise ValueError("Mismatch between CV X/M and y lengths")

    if not np.isfinite(fold_cv_raw).all():
        raise ValueError("fold.npy contains NaN/inf; CV split must be defined for all rows")
    fold_cv = fold_cv_raw.astype(int)

    bad = ~np.isfinite(X_cv)
    if bad.any():
        print(f"[CV] Warning: {int(bad.sum())} non-finite X values set to 0.")
        X_cv[bad] = 0.0
    bad_m = ~np.isfinite(M_cv)
    if bad_m.any():
        print(f"[CV] Warning: {int(bad_m.sum())} non-finite M values set to 0.")
        M_cv[bad_m] = 0.0

    # ----------- load TEST data -----------
    X_test = np.load(TEST_X_PATH)
    M_test = np.load(TEST_M_PATH)
    y_test = np.load(TEST_Y_PATH).astype(np.float32)

    print("[TEST] X shape:", X_test.shape)
    print("[TEST] M shape:", M_test.shape)
    print("[TEST] y shape:", y_test.shape)

    if X_test.shape[0] != y_test.shape[0] or M_test.shape[0] != y_test.shape[0]:
        raise ValueError("Mismatch between TEST X/M and y lengths")

    bad = ~np.isfinite(X_test)
    if bad.any():
        print(f"[TEST] Warning: {int(bad.sum())} non-finite X values set to 0.")
        X_test[bad] = 0.0
    bad_m = ~np.isfinite(M_test)
    if bad_m.any():
        print(f"[TEST] Warning: {int(bad_m.sum())} non-finite M values set to 0.")
        M_test[bad_m] = 0.0

    # ----------- CV training over folds -----------
    fold_ids = make_cv_indices(fold_cv)
    print("Folds:", fold_ids)

    # OOF bounded scores (0..1) for metrics comparability
    oof_score01 = np.full_like(y_cv, np.nan, dtype=float)

    fold_aucs = []
    fold_boyces = []

    for fid in fold_ids:
        best_val, val_score01, val_idx = train_and_save_fold_model_kulsif_cnn(
            fid, X_cv, M_cv, y_cv, fold_cv, MODEL_DIR
        )
        fold_aucs.append(best_val["ROC_AUC"])
        fold_boyces.append(best_val["Boyce"])
        oof_score01[val_idx] = val_score01

    # ----------- CV metrics (OOF bounded score) -----------
    cv_metrics = compute_boyce_and_auc(y_cv, oof_score01)
    print("\n[CV] OOF metrics (bounded score = w/(1+w)):", cv_metrics)
    print("[CV] per-fold best AUCs:", fold_aucs)
    print("[CV] per-fold best Boyce:", fold_boyces)

    # ----------- Ensemble weights -----------
    # KuLSIF doesn't give a comparable "loss" like BCE, so default to equal weights.
    weights = np.ones(len(fold_ids), dtype=float) / len(fold_ids)
    print("\n[Ensemble] Using equal weights:", weights)

    np.save(os.path.join(MODEL_DIR, "fold_ids.npy"), np.array(fold_ids, dtype=int))
    np.save(os.path.join(MODEL_DIR, "fold_weights_equal.npy"), weights)
    np.save(os.path.join(MODEL_DIR, "oof_score01.npy"), oof_score01)
    print("[Saved] fold_ids.npy, fold_weights_equal.npy, oof_score01.npy")

    # ----------- External test ensemble (bounded score) -----------
    test_idx = np.arange(len(y_test))
    print("External test N =", len(test_idx))

    test_score01, test_w, test_y = ensemble_predict_on_indices_kulsif_cnn(
        X_values=X_test,
        X_masks=M_test,
        y=y_test,
        indices=test_idx,
        model_dir=MODEL_DIR,
        fold_ids=fold_ids,
        weights=weights,
    )

    test_metrics = compute_boyce_and_auc(test_y, test_score01)
    print("\n[Test] Ensemble metrics (bounded score = w/(1+w)):", test_metrics)

    np.save(os.path.join(MODEL_DIR, "test_score01.npy"), test_score01)
    np.save(os.path.join(MODEL_DIR, "test_w_ens.npy"), test_w)
    print("[Test] Saved test_score01.npy, test_w_ens.npy")
